# Lesson 05 · Gradient descent: walking downhill step by step, from one curve to MNIST

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/05-gradient-descent-en.ipynb)

This notebook accompanies [Lesson 05 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/gradient-descent/), a deep-learning course originally in
Vietnamese. It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/05-gradient-descent.ipynb)):
the code, the data and every check are the same. The reel and the interactive demo are on the lesson page.
Lesson 04 gave us a number that measures how wrong a model is: the loss function $\mathcal{L}(\theta)$. This lesson
answers the next question: **in which direction should we change the parameters $\theta$, and by how big a step, to
make that number smaller?** We write gradient descent ourselves in numpy, first on loss functions you could draw by
hand, then on real MNIST images.

Section 0 sets out all of the lesson's theory first, without running anything; the sections after it rebuild each
idea in numpy and check it with numbers.

**What you'll do:**
1. Measure a gradient with finite differences, and use it to check a hand-derived derivative formula.
2. Run $\theta \leftarrow \theta - \eta\,\nabla\mathcal{L}(\theta)$ on the demo's loss function with four learning
   rates, and find the threshold $\eta < 2/\lambda$.
3. See why the gradient doesn't point straight at the bottom, and why gradient descent can come to a standstill at a
   saddle point.
4. Train softmax regression on MNIST: gradient descent on the whole training set against mini-batch SGD, steps
   against epochs.
5. Measure the noise of the batch gradient as a function of the batch size $B$, then train the course's
   784 → 128 → 64 → 10 network with the same loop.

**Background:** [Lesson 04 · Softmax & loss functions](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/ham-mat-mat/)
(softmax, cross-entropy) and the convention $z = xW + b$ from [Lesson 01 · Perceptron](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/perceptron/).
Derivatives at the level of “the slope of the tangent line” are enough for sections 1–13; section 0 also uses Taylor
expansions, the eigenvalues of a symmetric matrix, and expectations and variances. **Run time:** under a minute on a
CPU, including the first MNIST download (~11 MB).

Every number in the notebook is computed by the cells below, and matches the numbers in the demo and in the course's
training data.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 0 · Theory before code

This section gathers all of the lesson's theory in one place, and runs no cells yet. Sections 1–13 rebuild each idea
in numpy, first on small functions you can draw, then on real MNIST images; the table at the end of this section says
where each one is checked. Notation: $\theta$ holds all the parameters, $\mathcal{L}(\theta)$ is the loss function,
$\theta_k$ is the value after $k$ steps and $\eta > 0$ is the learning rate; $g = \nabla\mathcal{L}(\theta)$ is the
gradient, and $H$ is the Hessian matrix (the second derivatives $\partial^2\mathcal{L}/\partial\theta_i\partial\theta_j$).
As everywhere in the course, vectors are row vectors ($z = xW + b$), so the dot product of $g$ and $\delta$ is written
$g\,\delta^\top$.

### 0.1 · Derivatives, the gradient and the steepest direction

With one parameter, the derivative $\mathcal{L}'(\theta)$ is the slope of the tangent line: as $\delta \to 0$,
$\mathcal{L}(\theta + \delta) = \mathcal{L}(\theta) + \mathcal{L}'(\theta)\,\delta + o(\delta)$. With $n$ parameters, the
**partial derivative** $\partial\mathcal{L}/\partial\theta_i$ measures how fast $\mathcal{L}$ changes when only
$\theta_i$ is nudged. Stacking the $n$ partial derivatives into a vector gives the **gradient**, and the first-order
Taylor expansion:

$$\nabla\mathcal{L}(\theta) = \Big(\frac{\partial\mathcal{L}}{\partial\theta_1}, \dots, \frac{\partial\mathcal{L}}{\partial\theta_n}\Big),
\qquad \mathcal{L}(\theta + \delta) = \mathcal{L}(\theta) + \nabla\mathcal{L}(\theta)\,\delta^\top + o(\lVert\delta\rVert).$$

**Why the gradient points in the direction of steepest ascent.** Moving a distance $t$ along a unit direction $u$
($\lVert u\rVert = 1$), the loss changes at the rate

$$\lim_{t \to 0}\frac{\mathcal{L}(\theta + t u) - \mathcal{L}(\theta)}{t} = \nabla\mathcal{L}(\theta)\,u^\top.$$

By the Cauchy–Schwarz inequality, $-\lVert\nabla\mathcal{L}\rVert \le \nabla\mathcal{L}\,u^\top \le \lVert\nabla\mathcal{L}\rVert$,
and (when $\nabla\mathcal{L} \ne 0$) equality holds only for $u = \pm\nabla\mathcal{L}/\lVert\nabla\mathcal{L}\rVert$. So
$\nabla\mathcal{L}$ is the direction in which the loss increases fastest, at the rate $\lVert\nabla\mathcal{L}\rVert$,
and $-\nabla\mathcal{L}$ is the direction in which it decreases fastest. Along every direction perpendicular to the
gradient, the loss doesn't change to first order: the gradient is perpendicular to the contour line through $\theta$.
“Fastest” here means at $\theta$, for a very short step.

**Measuring a derivative with finite differences.** Expand $\mathcal{L}(\theta + h)$ and $\mathcal{L}(\theta - h)$
around $\theta$; in their difference, the even-order terms cancel:

$$\frac{\mathcal{L}(\theta + h) - \mathcal{L}(\theta - h)}{2h} = \mathcal{L}'(\theta) + \frac{h^2}{6}\,\mathcal{L}'''(\theta) + O(h^4).$$

The error of this **central difference** is proportional to $h^2$ (the one-sided difference
$(\mathcal{L}(\theta + h) - \mathcal{L}(\theta))/h$ is only proportional to $h$). But a computer stores each value of
$\mathcal{L}$ with a relative error of about $\epsilon_{\mathrm{mach}} \approx 10^{-16}$, so the subtraction adds an error
of about $\epsilon_{\mathrm{mach}}\lvert\mathcal{L}\rvert/h$, which grows as $h$ shrinks. The sum
$\tfrac{h^2}{6}\lvert\mathcal{L}'''\rvert + \epsilon_{\mathrm{mach}}\lvert\mathcal{L}\rvert/h$ is smallest at
$h = (3\epsilon_{\mathrm{mach}}\lvert\mathcal{L}\rvert/\lvert\mathcal{L}'''\rvert)^{1/3}$, that is, of the order of
$\epsilon_{\mathrm{mach}}^{1/3}$, a few millionths, when $\mathcal{L}$ and $\mathcal{L}'''$ are of similar size. With
$n$ parameters, nudging each coordinate in turn costs $2n$ evaluations of $\mathcal{L}$: enough to check a gradient
formula, far too expensive for training.

**Bottoms and peaks in one dimension.** At a point where $\mathcal{L}'(\theta^*) = 0$,
$\mathcal{L}(\theta^* + \delta) = \mathcal{L}(\theta^*) + \tfrac12\mathcal{L}''(\theta^*)\,\delta^2 + o(\delta^2)$, so
$\mathcal{L}''(\theta^*) > 0$ makes it a **minimum** (lower than every point close enough to it) and
$\mathcal{L}''(\theta^*) < 0$ a **maximum**; $\mathcal{L}''$ is the **curvature**. A function can have several minima:
a minimum that is only lower than the points near it is a **local minimum**, and the lowest minimum of the whole
function is the **global minimum**.

### 0.2 · One gradient descent step, and why it decreases the loss

Gradient descent (Cauchy, 1847) repeats exactly one step:

$$\theta_{k+1} = \theta_k - \eta\,\nabla\mathcal{L}(\theta_k).$$

The step has length $\eta\,\lVert\nabla\mathcal{L}(\theta_k)\rVert$: the learning rate is a factor multiplying the
slope, not the length of the step, so the steps shorten by themselves where the slope is gentle.

**Why a small step helps.** Substitute $\delta = -\eta g$, with $g = \nabla\mathcal{L}(\theta) \ne 0$, into the
first-order expansion of section 0.1:

$$\mathcal{L}(\theta - \eta g) = \mathcal{L}(\theta) - \eta\,\lVert g\rVert^2 + o(\eta).$$

The term $-\eta\lVert g\rVert^2$ is negative, so for every small enough $\eta$ the loss decreases, by about
$\eta\lVert g\rVert^2$. By section 0.1, to first order no step of the same length $\eta\lVert g\rVert$ decreases the
loss more.

**How small is “small enough”.** Let $\varphi(t) = \mathcal{L}(\theta - t g)$. Taylor's theorem with the Lagrange
remainder gives a $\tau \in (0, \eta)$ such that $\varphi(\eta) = \varphi(0) + \eta\,\varphi'(0) + \tfrac12\eta^2\varphi''(\tau)$,
with $\varphi'(0) = -\lVert g\rVert^2$ and $\varphi''(\tau) = g\,H(\theta - \tau g)\,g^\top$. For a unit vector $v$,
$v H v^\top$ is the **curvature along** $v$. If the curvature along every direction, at every point of the segment from
$\theta$ to $\theta - \eta g$, is at most $\ell$, then

$$\mathcal{L}(\theta - \eta g) \le \mathcal{L}(\theta) - \eta\Big(1 - \frac{\eta\ell}{2}\Big)\lVert g\rVert^2,$$

so the loss is guaranteed to decrease when $0 < \eta < 2/\ell$. With one parameter, expanding to second order gives
$\mathcal{L}(\theta - \eta g) \approx \mathcal{L}(\theta) - \eta g^2\big(1 - \tfrac12\eta\mathcal{L}''(\theta)\big)$: the
ratio of the actual decrease to $\eta g^2$ tends to 1 as $\eta \to 0$, and its shortfall from 1 is about
$\tfrac12\eta\mathcal{L}''(\theta)$, proportional to $\eta$.

### 0.3 · Near a minimum: the quadratic model and the threshold $\eta < 2/\lambda$

Near a minimum $\theta^*$ (where $\nabla\mathcal{L}(\theta^*) = 0$), let $e = \theta - \theta^*$. The second-order
expansion, with $H$ the (symmetric) Hessian at $\theta^*$:

$$\mathcal{L}(\theta^* + e) \approx \mathcal{L}(\theta^*) + \tfrac12\, e H e^\top, \qquad \nabla\mathcal{L}(\theta^* + e) \approx e H.$$

Substituting into one step: $e_{k+1} \approx e_k - \eta\, e_k H = e_k\,(I - \eta H)$. Seen close enough to a minimum
where every curvature is positive, every valley is a quadratic bowl, and gradient descent on it is a repeated matrix
multiplication.

**One dimension.** With $\lambda = \mathcal{L}''(\theta^*) > 0$: $e_{k+1} = (1 - \eta\lambda)\,e_k$, that is,
$e_k = (1 - \eta\lambda)^k e_0$. The distance to the bottom shrinks if and only if $\lvert 1 - \eta\lambda\rvert < 1$,
that is,

$$0 < \eta < \frac{2}{\lambda}.$$

Within that range there are three regimes: $0 < \eta\lambda < 1$ gives a positive factor ($\theta$ approaches the
bottom from one side); $\eta\lambda = 1$ gives the factor 0 (the bottom in one step, if the quadratic model were
exact); $1 < \eta\lambda < 2$ gives a negative factor ($\theta$ switches sides at every step but still closes in). At
$\eta\lambda = 2$ the factor is $-1$: $\theta$ jumps between $\theta^* \pm e_0$ forever. When $\eta\lambda > 2$, the
deviation grows at every step. The quadratic model only says that this minimum cannot hold $\theta$; where $\theta$
goes after that is decided by the rest of the function. Two minima with different curvatures have different
thresholds.

**Many dimensions.** $H$ is symmetric, so it has $n$ orthonormal eigenvectors $q_1, \dots, q_n$ (row vectors,
$q_i H = \lambda_i q_i$). Write $e = \sum_i c_i\, q_i$ with $c_i = e\, q_i^\top$; since
$q_i(I - \eta H) = (1 - \eta\lambda_i)\, q_i$,

$$c_{i,k+1} = (1 - \eta\lambda_i)\, c_{i,k}, \qquad i = 1, \dots, n:$$

$n$ independent one-dimensional problems, one factor per eigendirection. At a minimum where every $\lambda_i > 0$, a
single shared $\eta$ makes every direction shrink if and only if

$$\eta < \frac{2}{\lambda_{\max}}:$$

the most curved direction sets the threshold.

**A long, narrow valley.** Let the **condition number** be $\kappa = \lambda_{\max}/\lambda_{\min}$. The contour lines
$\tfrac12 e H e^\top = C$ are ellipses (in many dimensions, ellipsoids) with semi-axes $\sqrt{2C/\lambda_i}$ along
$q_i$, so the longest axis is $\sqrt{\kappa}$ times the shortest. The gradient $eH = \sum_i \lambda_i c_i\, q_i$
magnifies the components along the more curved directions, so $-\nabla\mathcal{L}$ points straight at the bottom if
and only if $e$ is an eigenvector of $H$ (in a round bowl, every $e$ is). And for every stable $\eta$, the gentlest
direction has the factor $1 - \eta\lambda_{\min} > 1 - 2/\kappa$: when $\kappa$ is large, each step removes less than
$2/\kappa$ of the remaining distance along the floor of the valley.

### 0.4 · Stationary points in many dimensions: minima, maxima and saddle points

A **stationary point** is a point where $\nabla\mathcal{L}(\theta^*) = 0$. Every stationary point is a fixed point of
gradient descent ($\theta^* - \eta \cdot 0 = \theta^*$, for every $\eta$), whether it is a bottom or not. Written in the
eigendirections as in section 0.3, $\mathcal{L}(\theta^* + e) \approx \mathcal{L}(\theta^*) + \tfrac12\sum_i \lambda_i c_i^2$.
So if every $\lambda_i > 0$, $\theta^*$ is a minimum; if every $\lambda_i < 0$, a maximum; if some $\lambda_i$ are
positive and some negative, a **saddle point**: curving up along some directions, curving down along others. (When
some $\lambda_i = 0$, the second-order expansion is not enough to decide.)

**Gradient descent near a saddle point.** The multiplication of section 0.3 still holds:
$c_{i,k+1} = (1 - \eta\lambda_i)\,c_{i,k}$. Along a direction with $\lambda_i < 0$, the factor is
$1 + \eta\lvert\lambda_i\rvert > 1$ for every $\eta > 0$: that component grows geometrically and $\theta$ leaves the
saddle point. But only if that component is nonzero: if it is exactly 0, and the gradient never creates it, $\theta$
slides straight into the saddle point and stays there. Going from $\lvert c_i\rvert = \varepsilon$ out to a distance
$d$ (still within the region where the quadratic model holds) takes about

$$k \approx \frac{\ln(d/\varepsilon)}{\ln(1 + \eta\lvert\lambda_i\rvert)}$$

steps, more and more as $\varepsilon \to 0$ (like $\ln(1/\varepsilon)$). All that time $\theta$ stays near the
stationary point, $\lVert\nabla\mathcal{L}\rVert$ is small and $\mathcal{L}$ barely changes: a **flat region** on the
loss curve. A little noise in the gradient, like the gradient of a mini-batch (section 0.6), makes that component
nonzero right from the start. Lee et al. (2016) proved that if every eigenvalue of the Hessian, at every point, is at
most $\ell$ in absolute value, then with $\eta < 1/\ell$ and a random starting point, gradient descent almost surely
does not converge to a stationary point that has a $\lambda_i < 0$; the price is the time spent on flat regions.

**The example of section 6.** $f(\theta) = \theta_1^2 + \tfrac14\theta_2^4 - \tfrac12\theta_2^2$ has
$\nabla f = (2\theta_1,\ \theta_2^3 - \theta_2)$ and the Hessian $\operatorname{diag}(2,\ 3\theta_2^2 - 1)$. It has three
stationary points: the origin, with Hessian $\operatorname{diag}(2, -1)$, is a saddle point; $(0, \pm 1)$, with Hessian
$\operatorname{diag}(2, 2)$ and $f = -\tfrac14$, are two minima. On the line $\theta_2 = 0$, the second component of the
gradient is 0, so a path that starts on that line stays on it. With $\eta = 0.1$, near the origin $\theta_2$ is
multiplied by $1 + \eta = 1.1$ at every step.

**In many dimensions.** A minimum needs *all* $n$ eigenvalues to be positive. If the sign of each eigenvalue were a
coin toss, the probability would be $2^{-n}$. For a random symmetric matrix $H = (A + A^\top)/\sqrt{2}$, with the
$A_{ij}$ independent and $\mathcal{N}(0, 1)$ (the GOE ensemble), the eigenvalues repel each other and the probability
is even smaller. For $n = 2$ it can be computed exactly: $H_{11}, H_{22} \sim \mathcal{N}(0, 2)$ and
$H_{12} \sim \mathcal{N}(0, 1)$ are independent, so $s = (H_{11} + H_{22})/2$, $t = (H_{11} - H_{22})/2$ and $H_{12}$
are independent and $\mathcal{N}(0, 1)$, and the two eigenvalues are $s \pm r$ with $r = \sqrt{t^2 + H_{12}^2}$. Both
are positive when $s > r$. $r$ has the density $r\,e^{-r^2/2}$ on $r > 0$, so integrating by parts, with $\Phi$ the
distribution function of $\mathcal{N}(0, 1)$, gives

$$P(s > r) = \int_0^\infty r\,e^{-r^2/2}\,\big(1 - \Phi(r)\big)\,dr = \frac12 - \int_0^\infty \frac{e^{-r^2}}{\sqrt{2\pi}}\,dr
= \frac12 - \frac{1}{2\sqrt{2}} = \frac{2 - \sqrt{2}}{4} < \frac14.$$

For large $n$, this probability decreases like $e^{-\frac{\ln 3}{4}\,n^2}$ (Dean & Majumdar, 2006), far faster than
$2^{-n}$. Dauphin et al. (2014) argue that in a network with many parameters, most stationary points in high-loss
regions are saddle points; that is an argument about random functions, and for real networks it is still a hypothesis
under study.

### 0.5 · Softmax regression on MNIST: gradient and curvature

The model is one layer $z = xW + b$ (Lesson 01), $W$ of size $784 \times 10$ and $b$ of size $1 \times 10$, followed by
softmax and cross-entropy (Lesson 04). For an image $x$ whose correct label is $y$:

$$\hat y = \operatorname{softmax}(z), \qquad \ell = -\log\hat y_y = \log\sum_{j} e^{z_j} - z_y, \qquad
\mathcal{L} = \frac1N\sum_{i=1}^{N}\ell_i.$$

**The gradient.** From here on, $y$ is also the one-hot vector of the label ($y_k = 1$ at the correct digit, 0
elsewhere). The derivative with respect to the logits is
$\partial\ell/\partial z_k = e^{z_k}/\sum_j e^{z_j} - y_k = \hat y_k - y_k$ (Lesson 04): the predicted probability minus
the answer. Since $z_k = \sum_i x_i W_{ik} + b_k$, the chain rule through exactly one layer gives
$\partial\ell/\partial W_{ik} = x_i\,(\hat y_k - y_k)$ and $\partial\ell/\partial b_k = \hat y_k - y_k$. Stacking the
$N$ images as the rows of $X$, $\hat Y$ and $Y$:

$$\nabla_W\mathcal{L} = \frac1N X^\top(\hat Y - Y), \qquad \nabla_b\mathcal{L} = \frac1N\sum_{i=1}^{N}(\hat y_i - y_i).$$

**The gradient of one image is bounded.** Fold $b$ into $W$ with $\tilde x = [x, 1]$: $\tilde W$ is $W$ with the row
$b$ added at the bottom (size $785 \times 10$). The gradient of one image is the outer product
$\tilde x^\top(\hat y - y)$, a rank-1 matrix, so $\lVert\nabla\ell\rVert^2 = \lVert\tilde x\rVert^2\,\lVert\hat y - y\rVert^2$.
And since $\sum_{k \ne y}\hat y_k^2 \le \big(\sum_{k \ne y}\hat y_k\big)^2 = (1 - \hat y_y)^2$,

$$\lVert\hat y - y\rVert^2 = (1 - \hat y_y)^2 + \sum_{k \ne y}\hat y_k^2 \le 2\,(1 - \hat y_y)^2 \le 2:$$

the gradient of one image is never longer than $\sqrt{2}\,\lVert\tilde x\rVert$, however large the logits are.

**The curvature, and why this loss is convex.** The second derivative with respect to the logits is
$\partial^2\ell/\partial z_k\partial z_l = \hat y_k\,[k = l] - \hat y_k\hat y_l$ (where $[k = l]$ is 1 when $k = l$ and 0
otherwise), that is, the matrix $S = \operatorname{diag}(\hat y) - \hat y^\top\hat y$. For every row vector $v$,
$v S v^\top = \sum_k \hat y_k v_k^2 - \big(\sum_k \hat y_k v_k\big)^2 \ge 0$: it is the variance of $v_k$ when the index
$k$ is drawn with the probabilities $\hat y$. Moving along a direction $V$ (of size $785 \times 10$) in parameter
space, the logits of image $i$ change linearly, $z_i(t) = \tilde x_i(\tilde W + tV)$, so the curvature along that
direction is ($S_i$ is the $S$ of image $i$)

$$\frac{d^2}{dt^2}\,\mathcal{L}(\tilde W + tV) = \frac1N\sum_{i=1}^{N}(\tilde x_i V)\,S_i\,(\tilde x_i V)^\top \ge 0.$$

The curvature is nonnegative along every direction, at every point: $\mathcal{L}$ is a **convex** function of
$(W, b)$. For a differentiable convex function, every point with zero gradient is a global minimum, so this problem
has no saddle points and no bad local minima.

**At $W = 0$.** When $W = 0$ and $b = 0$, every image has $\hat y_i = \tfrac{1}{10}\mathbf{1}$ (ten equal probabilities,
so $\mathcal{L} = \ln 10$), and every $S_i$ is the same matrix
$S_0 = \tfrac{1}{10}I - \tfrac{1}{100}\mathbf{1}^\top\mathbf{1}$. Let $M = \tilde X^\top\tilde X/N$ (of size
$785 \times 785$). For an outer-product direction $V = u^\top c$ ($u$ holds 785 numbers, $c$ holds 10),
$\tilde x_i V = (\tilde x_i u^\top)\,c$, so the curvature above becomes

$$\frac1N\sum_{i=1}^{N}(\tilde x_i u^\top)^2\; c\,S_0\,c^\top = (u M u^\top)\,(c\,S_0\,c^\top).$$

The Hessian at $W = 0$ is the Kronecker product $M \otimes S_0$, so its eigenvalues are the products $\mu_j s$ of an
eigenvalue $\mu_j$ of $M$ and an eigenvalue $s$ of $S_0$. $S_0$ has the eigenvalue $\tfrac{1}{10}$ for every $c$ whose
entries sum to 0, and $0$ for $c = \mathbf{1}$ (adding the same number to every logit leaves the softmax unchanged).
So, with $\mu_{\max}$ the largest eigenvalue of $M$,

$$\lambda_{\max} = \frac{\mu_{\max}}{10}, \qquad \frac{2}{\lambda_{\max}} = \frac{20}{\mu_{\max}},$$

reached along the direction $V = u^\top c$ with $u$ the eigenvector of $\mu_{\max}$ and any $c$ that sums to 0. This is
only the curvature at $W = 0$: as $W$ changes, the $\hat y_i$ and the $S_i$ change with it.

### 0.6 · The gradient of the whole training set and of a batch: SGD

The loss is an average, so the gradient is too: $g = \tfrac1N\sum_i g_i$ with $g_i = \nabla\ell_i(\theta)$. Gradient
descent **on the whole training set** computes all $N$ per-image gradients at every step. **Mini-batch SGD** (which
goes back to the stochastic approximation method of Robbins and Monro, 1951) draws a batch $S$ of $B$ indices,
uniformly at random and without repetition, then follows the batch's gradient:

$$g_B = \frac1B\sum_{i \in S} g_i, \qquad \theta \leftarrow \theta - \eta\, g_B.$$

**Unbiased.** Each index is in the batch with probability $B/N$, so for a fixed $\theta$,
$\mathbb{E}[g_B] = \tfrac1B\sum_{i=1}^{N}\tfrac{B}{N}\, g_i = g$: $g_B$ is an **unbiased estimate** of the true
gradient.

**The variance.** Let $d_i = g_i - g$ (so $\sum_i d_i = 0$) and
$\sigma^2 = \tfrac1N\sum_i\lVert d_i\rVert^2 = \tfrac1N\sum_i\lVert g_i\rVert^2 - \lVert g\rVert^2$, the variance of the
gradient of one image. Since $g_B - g = \tfrac1B\sum_{i \in S} d_i$, and two different indices are both in the batch
with probability $\tfrac{B(B-1)}{N(N-1)}$,

$$\mathbb{E}\lVert g_B - g\rVert^2 = \frac{1}{B^2}\Big[\frac{B}{N}\sum_i\lVert d_i\rVert^2 + \frac{B(B-1)}{N(N-1)}\sum_{i \ne j} d_i\, d_j^\top\Big]
= \frac{\sigma^2}{B}\cdot\frac{N - B}{N - 1},$$

because $\sum_{i \ne j} d_i\, d_j^\top = \lVert\sum_i d_i\rVert^2 - \sum_i\lVert d_i\rVert^2 = -N\sigma^2$. When
$B \ll N$, the typical deviation $\sqrt{\mathbb{E}\lVert g_B - g\rVert^2}$ is about $\sigma/\sqrt{B}$: halving the noise
takes four times the batch size, and $B = N$ gives zero noise. The average of $k$ independently drawn batches is still
unbiased, with $k$ times less variance. For softmax regression,
$\lVert g_i\rVert^2 = \lVert\tilde x_i\rVert^2\,\lVert\hat y_i - y_i\rVert^2$ (section 0.5), so $\sigma^2$ can be computed
exactly with one pass over the data.

**Steps and epochs.** An **epoch** goes through all $N$ images once: shuffle the order, cut it into
$\lceil N/B\rceil$ consecutive batches (the last one has $N - B(\lceil N/B\rceil - 1)$ images), one **step** per batch.
Gradient descent on the whole training set is the case $B = N$: one step per epoch. A step costs $B$ per-image
gradients, so an epoch costs $N$, whatever $B$ is. Within an epoch the batches are not independent (an image that has
been used doesn't come back until the next epoch), but each full batch, taken on its own, is still $B$ images drawn
uniformly at random, so for a fixed $\theta$ it is still unbiased.

### 0.7 · The limit: the gradient of a multilayer network

Gradient descent needs only $\nabla\mathcal{L}(\theta)$, and doesn't care where it comes from. For softmax
regression, section 0.5 could write it down with one step of the chain rule. The course's network,
784 → 128 → 64 → 10,

$$h_1 = \operatorname{ReLU}(xW_1 + b_1), \qquad h_2 = \operatorname{ReLU}(h_1 W_2 + b_2), \qquad z = h_2 W_3 + b_3,$$

is different: $W_1$ affects $\ell$ only through $h_1$, $h_2$ and then $z$, so $\partial\ell/\partial W_1$ needs the
chain rule through all three layers. Lesson 06 (backpropagation) derives the whole gradient at roughly the cost of
one forward pass and one backward pass. The finite differences of section 0.1, on the other hand, cost $2P$
evaluations of $\mathcal{L}$ per gradient, with $P$ the number of parameters, so they are only good for checking
individual partial derivatives.

The loss is no longer convex either. The smallest example: two linear “layers” with one weight each,
$\hat y = w_2 w_1 x$, with a single example $x = y = 1$ and the squared loss $\mathcal{L} = (w_1 w_2 - 1)^2$. At the
origin, $\nabla\mathcal{L} = \big(2(w_1 w_2 - 1)\,w_2,\ 2(w_1 w_2 - 1)\,w_1\big) = 0$, while
$\partial^2\mathcal{L}/\partial w_1^2 = 2w_2^2 = 0$, $\partial^2\mathcal{L}/\partial w_2^2 = 2w_1^2 = 0$ and
$\partial^2\mathcal{L}/\partial w_1\partial w_2 = 4w_1 w_2 - 2 = -2$: the Hessian has eigenvalues $\pm 2$, a saddle
point, which a convex function cannot have. The “convex” conclusion of section 0.5 does not carry over to multilayer
networks; what section 0.4 says about saddle points does. Finally, a single shared $\eta$ must stay below
$2/\lambda_{\max}$, so it moves slowly along the gentle directions (section 0.3); momentum, Adam and learning-rate
decay schedules (Lesson 07) target exactly that.

### Where each piece of theory is checked

| Theory | Checked with numbers in |
|---|---|
| 0.1 the gradient, the steepest direction, the central difference with an error $\propto h^2$ | section 3 (one variable), section 5 (the gradient misses the bottom), sections 7 and 11 (gradient formulas against finite differences) |
| 0.2 a small step decreases $\mathcal{L}$ by about $\eta\lVert g\rVert^2$, the bound $\eta(1 - \eta\ell/2)\lVert g\rVert^2$ | section 4 |
| 0.3 the factor $1 - \eta\lambda$, the threshold $2/\lambda_{\max}$, the narrow bowl | section 4 (four learning rates), section 5 (the measured factor, the sweep of $\eta$, the bowl $\lambda = (1, 10)$), section 8 (MNIST) |
| 0.4 saddle points, the escape time $\ln(d/\varepsilon)/\ln(1 + \eta\lvert\lambda\rvert)$, the eigenvalues of random matrices | section 6 |
| 0.5 the gradient $X^\top(\hat Y - Y)/N$, $\lambda_{\max} = \mu_{\max}/10$ at $W = 0$ | sections 7, 8 |
| 0.6 $g_B$ unbiased, $\mathbb{E}\lVert g_B - g\rVert^2 = \tfrac{\sigma^2}{B}\cdot\tfrac{N - B}{N - 1}$, steps and epochs | sections 9, 10 |
| 0.7 the gradient of a multilayer network | section 11 (the backward pass against finite differences), section 13 |

## 1 · Setup

Only numpy and matplotlib are needed (Colab has them preinstalled). The function `num()` prints numbers in the
English format: a decimal point, and commas between thousands; `sci()` prints very small numbers in the form
3.2·10⁻¹¹. That is the same point Python uses when you *type* a decimal (`0.1`); printing only adds the thousands
separators and a real minus sign.

In [ ]:
import gzip
import hashlib
import itertools
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FuncFormatter, ScalarFormatter

# The course's palette (colorblind-safe), always paired with a second encoding (marker shape, dash style, label)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def num(x, digits=2):
    """A number in the English format: 13,007 · 0.1 · −4.38."""
    return f"{x:,.{digits}f}".replace("-", "−")


def pct(p, digits=2):
    return num(100 * p, digits) + "%"


def tick(v, _pos=None):
    """An axis tick label: 0.5 · −2 · 60,000 (no trailing zeros)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return num(v, len(s.split(".")[1]) if "." in s else 0)


def fmt_axes(fig):
    """Put the tick labels of every linear axis (colorbars included) in the English format; call right before
    plt.show()."""
    for ax in fig.axes:
        for axis in (ax.xaxis, ax.yaxis):
            # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
            if axis.get_scale() == "linear" and isinstance(axis.get_major_formatter(), ScalarFormatter):
                axis.set_major_formatter(FuncFormatter(tick))


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def sci(x, digits=1):
    """A very small or very large number: 3.2·10⁻¹¹."""
    m, e = f"{x:.{digits}e}".split("e")
    return f"{num(float(m), digits)}·10{str(int(e)).translate(SUP)}"


print(num(60000, 0), num(0.15), num(-2.0879, 3), pct(0.9089), sci(3.2e-11))

## 2 · The MNIST data

Sections 3–6 run on small loss functions given by a formula; from section 7 on we use MNIST (LeCun et al., 1998):
grayscale 28×28 images of handwritten digits. The cell below downloads the 4 original files and checks their MD5
hashes as in Lesson 01, then lays out each image as a row $x$ of 784 numbers in $[0, 1]$. The whole training set is the
matrix $X$: one image per row.

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # MNIST's standard MD5 hashes
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"


def load(kind):
    """IDX format: the first 16 bytes of an image file (8 bytes of a label file) are the header; the rest are bytes
    0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


fetch()
train_img, train_lab = load("train")
test_img, test_lab = load("t10k")
Xtr, ytr = train_img.reshape(-1, 784) / 255.0, train_lab.astype(int)  # each row is one image x (a 1×784 row vector)
Xte, yte = test_img.reshape(-1, 784) / 255.0, test_lab.astype(int)
N = len(Xtr)
print(f"training: {num(N, 0)} images · test: {num(len(Xte), 0)} images · total {num(N + len(Xte), 0)}")
print("the training X has shape", Xtr.shape)
assert (N, len(Xte)) == (60000, 10000)

The first ten training images, each with its true label above it:

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(10, 1.4))
for ax, img, lab in zip(axes, train_img, train_lab):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(str(lab), fontsize=11)
    ax.axis("off")
fmt_axes(plt.gcf())
plt.show()

## 3 · The slope at a point is the gradient

Imagine a hiker in the fog: they can't see where the bottom of the valley is, and can only feel which way the ground
under their feet slopes up. Gradient descent is the same: it only knows **the slope at the point where it stands**.

The lesson's demo uses a one-dimensional loss function with two valleys:

$$\mathcal{L}(\theta) = 0.1\,\theta^4 - 0.8\,\theta^2 + 0.3\,\theta + 2.5, \qquad
\mathcal{L}'(\theta) = 0.4\,\theta^3 - 1.6\,\theta + 0.3.$$

The derivative $\mathcal{L}'(\theta)$ is the slope of the tangent line (section 0.1). We can measure it even without a formula:
nudge $\theta$ by a small amount $h$ to either side and see how much $\mathcal{L}$ changes (the **central
difference**):

$$\mathcal{L}'(\theta) \approx \frac{\mathcal{L}(\theta + h) - \mathcal{L}(\theta - h)}{2h}.$$

With many parameters $\theta = (\theta_1, \dots, \theta_n)$, nudging each coordinate in turn gives $n$ **partial
derivatives**; stacked into a vector, they form the **gradient** $\nabla\mathcal{L}(\theta)$. The function `num_grad`
below does this for any array; the whole lesson uses it to check every hand-derived gradient formula.

In [ ]:
def L(t):
    return 0.1 * t**4 - 0.8 * t**2 + 0.3 * t + 2.5  # the demo's loss function


def dL(t):
    return 0.4 * t**3 - 1.6 * t + 0.3  # the derivative, derived by hand


def num_grad(f, theta, h=1e-5):
    """The gradient by central differences: nudge each coordinate of theta in turn to either side."""
    theta = np.array(theta, dtype=float)
    g = np.zeros_like(theta)
    for i in range(theta.size):
        e = np.zeros_like(theta)
        e.flat[i] = h
        g.flat[i] = (f(theta + e) - f(theta - e)) / (2 * h)
    return g


def L_vec(t):
    return L(t[0])


t0 = 2.6  # the demo's default starting point
g_num = num_grad(L_vec, [t0])[0]
print(
    f"θ = {num(t0, 1)}: formula L′ = {num(dL(t0), 4)}, central difference = {num(g_num, 4)} (off by {sci(abs(g_num - dL(t0)))})"
)
assert abs(g_num - dL(t0)) < 1e-8

The difference matches the formula to 10 digits. Why does `num_grad` use $h = 10^{-5}$? The cell below tries several
values of $h$:

In [ ]:
for h in [1e-1, 1e-3, 1e-5, 1e-7, 1e-9, 1e-11]:
    err = abs((L(t0 + h) - L(t0 - h)) / (2 * h) - dL(t0))
    print(f"h = {sci(h, 0):>6}:  error {sci(err)}")

The error shrinks like $h^2$ as $h$ gets smaller (each time $h$ is divided by 100, the error drops about
10,000-fold), then **grows** again when $h$ is too small: a computer's floating-point numbers keep only about 16
digits, so $\mathcal{L}(\theta + h) - \mathcal{L}(\theta - h)$ loses almost all of its significant digits. Smaller $h$
is not always better; $10^{-5}$ sits in the middle.

Bottoms and peaks are where the slope is 0. Solving $\mathcal{L}'(\theta) = 0$ (a cubic polynomial) gives three
points; the sign of the second derivative $\mathcal{L}''(\theta) = 1.2\,\theta^2 - 1.6$ (the curvature) tells whether
each one is a bottom or a peak.

In [ ]:
roots = np.sort(np.roots([0.4, 0, -1.6, 0.3]).real)  # the three roots of L′(θ) = 0
d2 = 1.2 * roots**2 - 1.6  # L″(θ): positive is a bottom, negative a peak
for r, c in zip(roots, d2):
    print(f"θ = {num(r, 3):>6}:  L = {num(L(r), 3)},  L″ = {num(c, 3):>6}  →  {'minimum' if c > 0 else 'maximum'}")
assert np.round(roots[[0, 2]], 3).tolist() == [-2.088, 1.899], "differs from the demo: check the function L"

Exactly the two bottoms the demo marks: the **global minimum** $\theta \approx -2.088$ (the lowest loss) and the
**local minimum** $\theta \approx 1.899$, separated by a peak at $\theta \approx 0.189$.

The figure below shows $\mathcal{L}(\theta)$ (blue curve), the tangent lines at four points (gray dashes) and orange
arrows pointing in the direction $-\mathcal{L}'$, that is, downhill. The global minimum is a circle, the local
minimum a square, the peak a triangle.

In [ ]:
grid = np.linspace(-3, 3, 400)
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(grid, L(grid), color=BLUE, lw=2.5, label="L(θ)")
for p in [-2.6, -0.8, 1.0, 2.6]:
    s = dL(p)
    tt = np.array([p - 0.45, p + 0.45])
    ax.plot(tt, L(p) + s * (tt - p), color=GREY, ls="--", lw=1.5)
    ax.annotate(
        "", xy=(p - 0.5 * np.sign(s), L(p)), xytext=(p, L(p)), arrowprops={"arrowstyle": "->", "color": ORANGE, "lw": 2}
    )
    ax.text(p, L(p) + 0.35, f"L′ = {num(s, 2)}", ha="center", fontsize=9)
for r, m, name in zip(roots, ["o", "^", "s"], ["global minimum", "peak", "local minimum"]):
    ax.plot(r, L(r), m, color="black", ms=8)
    ax.text(r, L(r) - 0.45, name, ha="center", fontsize=9)
ax.set(xlabel="θ", ylabel="L(θ)", xlim=(-3, 3), ylim=(-0.3, 5), title="The slope at a point: the orange arrow is −L′")
fmt_axes(plt.gcf())
plt.show()
print("slope at four points:", " · ".join(f"L′({num(p, 1)}) = {num(dL(p), 2)}" for p in [-2.6, -0.8, 1.0, 2.6]))

A positive slope (at $\theta = 2.6$) means moving right is uphill, so we step left; a negative slope means the
opposite. Either way, stepping **against the sign** of the derivative goes downhill. The same holds in many
dimensions: $\nabla\mathcal{L}$ points in the direction in which the loss increases fastest, so we move along
$-\nabla\mathcal{L}$.

## 4 · One step downhill: $\theta \leftarrow \theta - \eta\,\mathcal{L}'(\theta)$

Gradient descent (Cauchy, 1847) repeats exactly one thing (the rule of section 0.2): standing at $\theta$, measure the slope, then step against
it by an amount proportional to the **learning rate** $\eta$. The step length is $\eta\,|\mathcal{L}'(\theta)|$: the
steeper the slope, the longer the step, and near the bottom the slope is gentle, so the steps shorten by themselves.
The function `gd_1d` stops by exactly the demo's rules: **converged** when $|\mathcal{L}'| < 10^{-3}$ or $\theta$
barely changes; **diverged** when $\theta$ flies out of the region $|\theta| \le 3.4$; and it stops after 80 steps.

In [ ]:
ESCAPE, MAX_STEPS = 3.4, 80  # as in the demo


def gd_1d(eta, theta0, max_steps=MAX_STEPS):
    """One-dimensional gradient descent. Returns the sequence θ₀, θ₁, … and the status when it stops."""
    trail = [theta0]
    while True:
        t = trail[-1]
        if abs(t) > ESCAPE:
            return np.array(trail), "diverged"
        if len(trail) > 1 and (abs(dL(t)) < 1e-3 or abs(t - trail[-2]) < 1e-5):
            return np.array(trail), "converged"
        if len(trail) > max_steps:
            return np.array(trail), "not stopped"
        trail.append(t - eta * dL(t))  # one gradient descent step


t1 = t0 - 0.15 * dL(t0)
print(f"first step with η = 0.15:  θ ← {num(t0, 3)} − 0.15 × {num(dL(t0), 3)} = {num(t1, 3)}")

### 🤔 Predict first
Start at $\theta_0 = 2.6$ with $\eta = 0.15$ (the demo's “Medium” button). Which bottom does gradient descent stop
at: the global minimum $\theta \approx -2.088$ or the local minimum $\theta \approx 1.899$?

In [ ]:
trail, status = gd_1d(0.15, 2.6)
print(f"from θ₀ = 2.6:  {status} after {len(trail) - 1} steps, θ = {num(trail[-1], 3)}, L = {num(L(trail[-1]), 3)}")
trail_neg, status_neg = gd_1d(0.15, -2.6)
print(
    f"from θ₀ = −2.6: {status_neg} after {len(trail_neg) - 1} steps, θ = {num(trail_neg[-1], 3)}, L = {num(L(trail_neg[-1]), 3)}"
)
assert (status, len(trail) - 1, round(trail[-1], 2)) == ("converged", 13, 1.9), "differs from the demo"

It stops at the **local minimum**, after 13 steps, exactly as the demo reports. The gradient is only local
information: from $\theta_0 = 2.6$, every slope along the way points to the nearest bottom, and nothing tells it that
beyond the peak at $0.189$ there is a deeper valley. Starting at $-2.6$, it reaches the global minimum. Gradient
descent finds *a* point where the gradient is 0, not the lowest point; the starting point decides which valley it
falls into.

### Checking section 0.2: how much does one step decrease $\mathcal{L}$?
By section 0.2, one step from $\theta$ decreases $\mathcal{L}$ by about $\eta\,\mathcal{L}'(\theta)^2$, more precisely by
$\eta\,\mathcal{L}'^2\big(1 - \tfrac12\eta\mathcal{L}''\big)$, and by no less than $\eta\big(1 - \tfrac12\eta\ell\big)\mathcal{L}'^2$
if the curvature along the segment the step covers is at most $\ell$. The cell below takes one step from
$\theta_0 = 2.6$ with $\eta = 0.15$ and three smaller learning rates, each 10 times smaller than the last. For each
$\eta$, it prints the measured decrease, its ratio to $\eta\,\mathcal{L}'^2$, and the ratio the second-order expansion
predicts, $1 - \tfrac12\eta\mathcal{L}''(\theta_0)$; the last line prints the ratio's shortfall from 1, divided by
$\eta$. Since $\mathcal{L}''(\theta) = 1.2\,\theta^2 - 1.6$ increases with $\theta > 0$, the largest curvature on each
segment is $\ell = \mathcal{L}''(\theta_0)$.

In [ ]:
# L′(θ₀) and L″(θ₀); L″ increases with θ > 0, so c0 is the ℓ of every segment [θ₀ − ηL′, θ₀] below
g0, c0 = dL(t0), 1.2 * t0**2 - 1.6
for eta in [0.15, 0.015, 0.0015, 0.00015]:
    drop = L(t0) - L(t0 - eta * g0)  # how much L decreases after one step
    ratio = drop / (eta * g0**2)
    print(
        f"η = {num(eta, 5).rstrip('0'):<7}  L decreases by {num(drop, 6)},  ratio to ηL′²: {num(ratio, 4)},  second-order prediction {num(1 - eta * c0 / 2, 4)}"
    )
    assert drop >= eta * (1 - eta * c0 / 2) * g0**2  # the lower bound of section 0.2
print(f"η = {num(eta, 5).rstrip('0')}:  (1 − ratio)/η = {num((1 - ratio) / eta, 4)},  L″(θ₀)/2 = {num(c0 / 2, 4)}")
assert abs((1 - ratio) / eta - c0 / 2) < 1e-3  # the shortfall is proportional to η, with the coefficient L″/2
assert abs(L(t0) - 2.4418) < 5e-5 and abs(L(t0 - 0.15 * g0) - 1.5637) < 5e-5, (
    "differs from the lesson page: check the data"
)

As $\eta$ shrinks, the ratio tends to 1: to first order, one step decreases $\mathcal{L}$ by exactly
$\eta\,\mathcal{L}'^2$. The shortfall from 1 shrinks in step with $\eta$ (from $\eta = 0.015$ down, each division by 10
makes it about 10 times smaller), and divided by $\eta$ it tends to $\mathcal{L}''(\theta_0)/2$, exactly as the
second-order expansion says. At $\eta = 0.15$ the second-order expansion is already off (0.5116 against the measured
0.5824): the step is long enough for the third- and fourth-order terms ($\mathcal{L}$ is a quartic) to contribute
noticeably. The lower bound holds at every $\eta$. The decrease of 0.878086 at $\eta = 0.15$ is exactly
$\mathcal{L}(\theta_0) - \mathcal{L}(\theta_1)$ in the lesson page's table of the first steps.

### Three learning rates: crawling, just right and blowing up
The demo has four buttons: Small 0.02 · Medium 0.15 · Large 0.9 · Too large 1.3. Run all four from the same
$\theta_0 = 2.6$:

In [ ]:
PRESETS = {0.02: "Small", 0.15: "Medium", 0.9: "Large", 1.3: "Too large"}  # the demo's four buttons
runs = {eta: gd_1d(eta, 2.6) for eta in PRESETS}
for eta, (tr, st) in runs.items():
    print(
        f"η = {num(eta, 2):>4} ({PRESETS[eta]:>9}): {st:>11} after {len(tr) - 1:>2} steps, final θ = {num(tr[-1], 3)}"
    )
summary = [(st, len(tr) - 1) for tr, st in runs.values()]
assert summary == [("not stopped", 80), ("converged", 13), ("not stopped", 80), ("diverged", 3)], (
    "differs from the demo"
)

Each panel of the figure below is one learning rate: the blue curve is $\mathcal{L}(\theta)$, the connected orange
dots are $\theta_0, \theta_1, \dots$ (the large dot is the final point). Points that fly out of the frame are not
drawn.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True, sharey=True)
for ax, (eta, (tr, st)) in zip(axes.flat, runs.items()):
    ax.plot(grid, L(grid), color=BLUE, lw=2)
    inside = np.abs(tr) <= 3
    ax.plot(tr[inside], L(tr[inside]), "-o", color=ORANGE, ms=4, lw=1, alpha=0.7)
    if abs(tr[-1]) <= 3:
        ax.plot(tr[-1], L(tr[-1]), "o", color=ORANGE, ms=10, mec="white")
    ax.set_title(f"η = {num(eta, 2)} ({PRESETS[eta]}): {st} after {len(tr) - 1} steps", fontsize=10)
    ax.set(xlim=(-3, 3), ylim=(0, 5))
for ax in axes[1]:
    ax.set_xlabel("θ")
for ax in axes[:, 0]:
    ax.set_ylabel("L(θ)")
plt.tight_layout()
fmt_axes(plt.gcf())
plt.show()

- **Small (0.02):** heads in the right direction, but after 80 steps it still hasn't reached the bottom (it needs
  126 steps, see the cell below). This is “crawling”.
- **Medium (0.15):** reaches the bottom in 13 steps.
- **Large (0.9):** doesn't diverge, but never stops either: $\theta$ jumps wildly across both valleys, then falls
  into a loop between two points on either side of the local minimum.
- **Too large (1.3):** flies out after only 3 steps: $2.6 \to -1.52 \to -3.24 \to 7.38$. This is “blowing up”.

The next section explains why the boundary between “just right” and “jumping back and forth” lies exactly where it
does.

In [ ]:
tr_slow, st_slow = gd_1d(0.02, 2.6, max_steps=10_000)
print(f"η = 0.02: {st_slow} after {len(tr_slow) - 1} steps")
tr_big, _ = gd_1d(0.9, 2.6, max_steps=4000)
tail = tr_big[-4:]
print("η = 0.9 after 4,000 steps, the last four θ:", " · ".join(num(t, 4) for t in tail))
assert np.allclose(tail[:2], tail[2:], atol=1e-9)  # θ repeats every 2 steps: a 2-point cycle
print("η = 1.3:", " → ".join(num(t, 2) for t in runs[1.3][0]))

With $\eta = 0.9$, after 4,000 steps $\theta$ still swaps between exactly two points, 1.3879 and 2.1540, forever.
Gradient descent is not guaranteed to go downhill: every step jumps over the bottom to the slope on the other side.

## 5 · Why $\eta < 2/\lambda$: near the bottom, every valley is a bowl

As in section 0.3: near a minimum $\theta^*$, the curve looks like a parabola: $\mathcal{L}'(\theta) \approx \lambda\,(\theta - \theta^*)$,
where $\lambda = \mathcal{L}''(\theta^*)$ is the curvature at the bottom. Substituting into one step:

$$\theta_{k+1} - \theta^* \approx (1 - \eta\lambda)\,(\theta_k - \theta^*).$$

Each step multiplies the distance to the bottom by $1 - \eta\lambda$. The distance shrinks when
$|1 - \eta\lambda| < 1$, that is, $0 < \eta < 2/\lambda$; fastest when $\eta = 1/\lambda$ (the factor is 0). When
$1 < \eta\lambda < 2$, the factor is negative: $\theta$ still reaches the bottom, but jumps back and forth. When
$\eta\lambda > 2$, every step pushes $\theta$ farther away. The cell below computes the threshold $2/\lambda$ for the
demo's two bottoms:

In [ ]:
lam_glob, lam_loc = d2[0], d2[2]  # the curvature at the global and at the local minimum
for name, lam in [("global minimum", lam_glob), ("local minimum", lam_loc)]:
    print(f"{name}: λ = L″ = {num(lam, 3)},  best η 1/λ = {num(1 / lam, 3)},  threshold 2/λ = {num(2 / lam, 3)}")
for eta in PRESETS:
    print(
        f"η = {num(eta, 2):>4}:  1 − ηλ at the local minimum = {num(1 - eta * lam_loc, 3):>6},  at the global minimum = {num(1 - eta * lam_glob, 3):>6}"
    )

The threshold is 0.551 at the global minimum (a narrower, more curved bottom) and 0.734 at the local minimum. The
“Large” button, 0.9, exceeds both thresholds: neither bottom can hold $\theta$, so it falls into the back-and-forth
loop we just saw. The “Medium” button, 0.15, gives a factor of 0.591: the distance to the bottom shrinks to 59% at
every step. The “Small” button, 0.02, gives 0.945: each step removes only $1 - 0.945 = 5.5\%$.

This theory is only an approximation near the bottom, so we check it on the demo's own function. First, a trap: the
demo's stopping rule ($|\mathcal{L}'| < 10^{-3}$) can be fooled.

In [ ]:
tr_fake, st_fake = gd_1d(0.79, 2.6, max_steps=3000)
t_after = tr_fake[-1]
for _ in range(50):
    t_after -= 0.79 * dL(t_after)
print(
    f"η = 0.79: gd_1d reports “{st_fake}” after {len(tr_fake) - 1} steps, at θ = {num(tr_fake[-1], 3)} (|L′| = {sci(abs(dL(tr_fake[-1])))})"
)
print(f"   but after 50 more steps θ = {num(t_after, 3)}: it only happened to pass close to a bottom")
assert st_fake == "converged" and abs(t_after - tr_fake[-1]) > 0.1

A small gradient at *one* step doesn't mean it has converged: $\theta$ is jumping around and happens to land right
next to a bottom, then the next step throws it away again. So the sweep below uses a stricter rule: run 20,000 steps
for every $\eta$ from 0.01 to 1 at once (numpy computes the whole array of $\eta$ in one operation), and count a run
as converged only when the final $\theta$ sits still at a bottom ($|\mathcal{L}'| < 10^{-6}$).

In [ ]:
etas = np.round(np.arange(0.01, 1.0001, 0.01), 2)
for t_start, lam in [(2.6, lam_loc), (-2.6, lam_glob)]:
    th = np.full(etas.shape, t_start)
    for _ in range(20_000):
        th = th - etas * dL(th)  # 100 gradient descent runs in parallel, one η per element
        th[np.abs(th) > ESCAPE] = np.nan  # once out of bounds, stop for good
    ok = np.abs(dL(th)) < 1e-6  # NaN gives False
    print(
        f"from θ₀ = {num(t_start, 1)}: converges for every η ≤ {num(etas[ok].max(), 2)} (to θ = {num(th[ok][-1], 3)}), "
        f"does not converge for any η from {num(etas[~ok].min(), 2)} to 1 (after {num(20_000, 0)} steps);  2/λ of that minimum = {num(2 / lam, 3)}"
    )
    assert ok[etas < 2 / lam].all() and not ok[etas > 2 / lam].any()  # the boundary is exactly at 2/λ

The measured boundary coincides exactly with $2/\lambda$ of the bottom that $\theta$ rolls into. The same
$\eta = 0.6$ is fine at the local minimum (0.6 < 0.734) but too large at the global minimum (0.6 > 0.551): **the
“right” learning rate depends on the curvature where we stand**; it is not a constant of the problem.

### Checking section 0.3: does each step multiply the distance to the bottom by $1 - \eta\lambda$?
The sweep checks the threshold; section 0.3 says more: near the bottom,
$\theta_{k+1} - \theta^* \approx (1 - \eta\lambda)(\theta_k - \theta^*)$, sign included. The cell below runs gradient
descent into each minimum with two learning rates, $\eta = 0.15$ (where $\eta\lambda < 1$ at both minima) and
$\eta = 0.5$ (where $1 < \eta\lambda < 2$), until $\lvert\theta_k - \theta^*\rvert < 10^{-6}$. It prints the number of
steps taken, the ratio $(\theta_{k+1} - \theta^*)/(\theta_k - \theta^*)$ measured at the next step, the theory's
factor $1 - \eta\lambda$, and how many times $\theta_k - \theta^*$ changes sign over the last 10 steps.

In [ ]:
# lam_page: the curvature at that minimum on the lesson page (05-gradient-descent.runs.json)
for name, t_start, star, lam, lam_page in [
    ("local minimum", 2.6, roots[2], lam_loc, 2.725987),
    ("global minimum", -2.6, roots[0], lam_glob, 3.631061),
]:
    for eta in [0.15, 0.5]:
        th, dev = t_start, [t_start - star]  # dev: the signed deviation θ_k − θ*
        while abs(dev[-1]) > 1e-6 and len(dev) <= 10_000:  # cap the steps, in case a new η keeps GD away from θ*
            th -= eta * dL(th)
            dev.append(th - star)
        assert abs(dev[-1]) <= 1e-6, "GD hasn't reached θ* after 10,000 steps: with this η, the minimum θ* is unstable"
        ratio = (th - eta * dL(th) - star) / dev[-1]  # one more step: (θ_{k+1} − θ*)/(θ_k − θ*)
        flips = int(np.sum(np.diff(np.sign(dev[-11:])) != 0))
        print(
            f"{name}, η = {num(eta, 2)}: after {len(dev) - 1:>2} steps, measured ratio {num(ratio, 4):>7}, 1 − ηλ = {num(1 - eta * lam, 4):>7}; "
            f"{flips} sign changes over the last 10 steps"
        )
        assert abs(ratio - (1 - eta * lam_page)) < 1e-4, "differs from the lesson page: check the data"
        assert flips == (10 if eta * lam > 1 else 0)

The measured ratio matches $1 - \eta\lambda$ to 4 digits in all four cases, sign included. With $\eta = 0.15$ the
factor is positive (0.5911 at the local minimum, the same factor 0.591 that the first cell of this section computed
for the “Medium” button), and $\theta$ approaches the bottom from one side, never changing sign. With $\eta = 0.5$ the
factor is negative (−0.3630 and −0.8155): $\theta$ switches sides at every step and still reaches the bottom, exactly
the regime $1 < \eta\lambda < 2$ of section 0.3.

### In two dimensions: a long, narrow bowl
With many parameters, the curvature differs from one direction to another (section 0.3). The simplest bowl has two curvatures,
$\lambda_1 = 1$ (gentle) and $\lambda_2 = 10$ (steep):

$$\mathcal{L}(\theta) = \tfrac{1}{2}\left(\lambda_1\theta_1^2 + \lambda_2\theta_2^2\right), \qquad
\nabla\mathcal{L}(\theta) = (\lambda_1\theta_1,\; \lambda_2\theta_2).$$

Each coordinate gets multiplied by its own $1 - \eta\lambda_i$. A single shared $\eta$ must be small enough for the
steepest direction: $\eta < 2/\lambda_{\max} = 0.2$. But then the gentle direction moves very slowly.

In [ ]:
LAM = np.array([1.0, 10.0])  # the bowl's two curvatures
TH0 = np.array([-8.0, 2.0])  # the starting point


def bowl(theta):
    return 0.5 * np.sum(LAM * theta**2)


def gd_bowl(eta, steps, theta0=TH0):
    path = [theta0.copy()]
    for _ in range(steps):
        path.append(path[-1] - eta * LAM * path[-1])  # θ ← θ − η∇L, with ∇L = λ ⊙ θ
    return np.array(path)


assert np.allclose(num_grad(bowl, TH0), LAM * TH0)  # the hand-derived gradient matches the finite differences
g = LAM * TH0
cos = (-g) @ (-TH0) / (np.linalg.norm(g) * np.linalg.norm(TH0))
print(
    f"at θ = (−8, 2): −∇L = ({num(-g[0], 0)}, {num(-g[1], 0)}), {num(np.degrees(np.arccos(cos)), 1)}° off the direction straight to the bottom"
)

**The gradient doesn't point at the bottom.** It points in the direction of steepest descent *right where it is*,
and in the narrow bowl that direction is 54.2° off the straight line to the bottom, because the steep wall
($\lambda_2 = 10$) pulls much harder. Only when every direction has the same curvature (a round bowl) does
$-\nabla\mathcal{L}$ point straight at the center.

### 🤔 Predict first
The threshold is $\eta < 0.2$. To shrink the distance to the bottom $\|\theta\|$ a million-fold, which $\eta$ needs
the fewest steps: a safe, small $\eta$ such as 0.1 (exactly $1/\lambda_2$, which wipes out the steep direction in a
single step), or the largest possible $\eta$, just under 0.2?

In [ ]:
def steps_to(eta, tol=1e-6, cap=20_000):
    """The number of steps for ‖θ‖ to shrink 1/tol-fold (None if it never does)."""
    th, n0 = TH0.copy(), np.linalg.norm(TH0)
    for k in range(cap):
        if np.linalg.norm(th) < tol * n0:
            return k
        th = th - eta * LAM * th
        if np.linalg.norm(th) > 1e6 * n0:
            return None
    return None


eta_best = 2 / (LAM.min() + LAM.max())  # = 2/11
for eta in [0.01, 0.05, 0.1, eta_best, 0.19, 0.199, 0.2, 0.21]:
    k = steps_to(eta)
    print(
        f"η = {num(eta, 3)}:  factors |1 − ηλ| = {num(abs(1 - eta * LAM[0]), 3)} and {num(abs(1 - eta * LAM[1]), 3)}  →  {num(k, 0) + ' steps' if k else 'never'}"
    )
kappa = LAM.max() / LAM.min()
assert steps_to(eta_best) == int(np.ceil(np.log(1e-6) / np.log((kappa - 1) / (kappa + 1)))) == 69
assert np.abs(gd_bowl(0.2, 40)[:, 1]).tolist() == [2.0] * 41  # η = 2/λ₂: θ₂ flips sign forever and never shrinks

Neither. $\eta = 0.1$ wipes out the steep direction in the very first step, but leaves the gentle direction shrinking
by a factor of only 0.9 per step: 131 steps. Near the threshold, the steep direction barely shrinks: $\eta = 0.199$
needs 1,234 steps, and at exactly $\eta = 0.2$, $\theta_2$ flips sign forever between $+2$ and $-2$. The best is
$\eta = 2/(\lambda_1 + \lambda_2) = 2/11 \approx 0.182$, where the two factors are equal in magnitude ($0.818$):
69 steps. That factor is $(\kappa - 1)/(\kappa + 1)$, with $\kappa = \lambda_{\max}/\lambda_{\min} = 10$ (the
condition number): the narrower the bowl, the slower gradient descent is, however cleverly $\eta$ is chosen.

The figure below shows the bowl's contour lines (gray ellipses, the bottom at the plus sign) and the first 25 steps
with four values of $\eta$. With 0.182, the path zig-zags back and forth across the steep walls while drifting slowly
along the narrow floor; with 0.21 it flies out of the frame.

In [ ]:
g1, g2 = np.meshgrid(np.linspace(-9, 9, 301), np.linspace(-3.5, 3.5, 201))
levels = [0.5 * k**2 for k in (0.5, 1, 2, 3, 4, 6, 8)]
fig, axes = plt.subplots(2, 2, figsize=(10, 5.6), sharex=True, sharey=True)
for ax, (eta, name) in zip(
    axes.flat, [(0.02, "too small"), (0.1, "1/λ₂"), (eta_best, "2/11, the fastest"), (0.21, "> 2/λ₂")]
):
    ax.contour(g1, g2, 0.5 * (LAM[0] * g1**2 + LAM[1] * g2**2), levels=levels, colors=GREY, linewidths=0.8)
    p = gd_bowl(eta, 25)
    p = p[np.all(np.abs(p) < [9, 3.5], axis=1)]
    ax.plot(p[:, 0], p[:, 1], "-o", color=ORANGE, ms=3.5, lw=1.2)
    ax.plot(*TH0, "s", color=BLUE, ms=8)
    ax.plot(0, 0, "+", color="black", ms=12, mew=2)
    ax.set_title(f"η = {num(eta, 3)} ({name})", fontsize=10)
for ax in axes[1]:
    ax.set_xlabel("θ₁ (curvature 1)")
for ax in axes[:, 0]:
    ax.set_ylabel("θ₂ (curvature 10)")
plt.tight_layout()
fmt_axes(plt.gcf())
plt.show()

(The blue square is the starting point, the orange dots are the steps.) The two axes are not on the same scale (the
vertical axis is stretched), so the valley looks less narrow than it really is. The next plot shows the number of
steps as a function of $\eta$ (log vertical axis): the bottom of the curve is at 2/11, and the dashed vertical line is
the threshold 0.2.

In [ ]:
eta_grid = np.linspace(0.004, 0.2, 50)[:-1]
ks = [steps_to(e) for e in eta_grid]
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.semilogy(eta_grid, ks, "-o", color=BLUE, ms=3)
ax.axvline(0.2, color=GREY, ls="--")
ax.plot(eta_best, steps_to(eta_best), "D", color=ORANGE, ms=9, label=f"η = 2/11: {steps_to(eta_best)} steps")
ax.set(
    xlabel="learning rate η", ylabel="steps for ‖θ‖ to shrink 10⁶-fold", title="The bowl λ = (1, 10): slow at both ends"
)
ax.legend(frameon=False)
fmt_axes(plt.gcf())
plt.show()

## 6 · Saddle points and flat regions

The gradient is 0 not only at a bottom (section 0.4). The two-dimensional function below has a **saddle point** at the origin:
along $\theta_1$ it is a bottom, along $\theta_2$ it is a peak (like a horse's saddle: curving up along the horse's
spine, curving down across it). The two real bottoms are at $(0, \pm 1)$, with $f = -0.25$.

$$f(\theta) = \theta_1^2 + \tfrac{1}{4}\theta_2^4 - \tfrac{1}{2}\theta_2^2, \qquad
\nabla f(\theta) = \left(2\theta_1,\; \theta_2^3 - \theta_2\right).$$

At the origin, the curvature is $+2$ along $\theta_1$ and $-1$ along $\theta_2$. We run gradient descent
($\eta = 0.1$) from $(2, 0)$, from a point $10^{-8}$ away from it, and with a little noise added to the gradient
(standard deviation 0.01), like the gradient of a mini-batch.

In [ ]:
def saddle(t):
    return t[0] ** 2 + t[1] ** 4 / 4 - t[1] ** 2 / 2


def saddle_grad(t):
    return np.array([2 * t[0], t[1] ** 3 - t[1]])


def gd_saddle(theta0, eta=0.1, steps=300, sigma=0.0, seed=0):
    rng = np.random.default_rng(seed)
    path = [np.array(theta0, dtype=float)]
    for _ in range(steps):
        g = saddle_grad(path[-1]) + sigma * rng.normal(size=2)  # sigma > 0: a noisy gradient
        path.append(path[-1] - eta * g)
    return np.array(path)


def escape_step(path, level=0.1):
    """The first step at which |θ₂| exceeds level, that is, θ has left the region around the saddle point."""
    far = np.abs(path[:, 1]) > level
    return int(np.argmax(far)) if far.any() else None


assert np.allclose(num_grad(saddle, [0.7, -0.4]), saddle_grad(np.array([0.7, -0.4])))  # the formula matches num_grad
exact = gd_saddle([2, 0])
tiny = gd_saddle([2, 1e-8])
print(
    f"from (2, 0):       after 300 steps θ = ({sci(exact[-1, 0])}, {num(exact[-1, 1], 1)}), f = {sci(saddle(exact[-1]))}"
)
print(f"from (2, 10⁻⁸):    leaves the saddle point at step {escape_step(tiny)}, final f = {num(saddle(tiny[-1]), 4)}")
pred = np.log(0.1 / 1e-8) / np.log(1 + 0.1 * 1)  # near the origin, θ₂ is multiplied by (1 + η·1) at every step
print(f"   prediction: θ₂ grows 1.1-fold per step, so from 10⁻⁸ to 0.1 takes ln(10⁷)/ln(1.1) = {num(pred, 1)} steps")
noisy = [escape_step(gd_saddle([2, 0], sigma=0.01, seed=s)) for s in range(5)]
print(f"from (2, 0) with noise σ = 0.01, 5 seeds: leaves the saddle point at steps {noisy}")
assert np.all(exact[:, 1] == 0) and abs(exact[-1, 0]) < 1e-20  # exactly on θ₂ = 0, the gradient never pushes θ₂
assert abs(escape_step(tiny) - pred) <= 2

Starting exactly on the line $\theta_2 = 0$, the second component of the gradient is always 0, so gradient descent
slides straight into the saddle point and stays there: $f \to 0$, well above the bottom's $-0.25$. Off by
$10^{-8}$, it does escape, but only after about 170 steps of wandering on a **flat region** near the origin, where
the gradient is almost 0 and each step multiplies $\theta_2$ by only $1.1$. A little noise pushes $\theta$ off the
dangerous line right from the start, so it escapes after 40–73 steps, depending on the seed.

The left plot shows the contour lines of $f$ (the saddle point is the cross, the two bottoms are the plus signs) and
the three paths; the right plot shows $f(\theta_k)$ against the step $k$. The horizontal stretch at $f \approx 0$ is
the flat region: looking at the loss curve, it's easy to believe training has converged.

In [ ]:
noisy0 = gd_saddle([2, 0], sigma=0.01, seed=0)
cases = [
    (exact, "from (2, 0)", GREY, "^", ":"),
    (tiny, "from (2, 10⁻⁸)", BLUE, "o", "-"),
    (noisy0, "from (2, 0), with noise", ORANGE, "s", "--"),
]
s1, s2 = np.meshgrid(np.linspace(-2.2, 2.2, 221), np.linspace(-1.6, 1.6, 161))
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.8), gridspec_kw={"width_ratios": [1.15, 1]})
a.contour(s1, s2, s1**2 + s2**4 / 4 - s2**2 / 2, levels=np.linspace(-0.24, 4, 14), colors=GREY, linewidths=0.7)
for path, label, color, marker, ls in cases:
    a.plot(path[:, 0], path[:, 1], ls=ls, color=color, lw=1.8, marker=marker, markevery=15, ms=5, label=label)
    b.plot(saddle(path.T), ls=ls, color=color, lw=2, marker=marker, markevery=15, ms=5, label=label)
a.plot(0, 0, "x", color="black", ms=10, mew=2)
a.plot([0, 0], [-1, 1], "+", color="black", ms=12, mew=2)
a.set(xlabel="θ₁", ylabel="θ₂", title="Paths around the saddle point (0, 0)")
b.axhline(-0.25, color="black", lw=0.8)
b.text(300, -0.2, "bottom: f = −0.25", ha="right", va="bottom", fontsize=9)  # above the row of markers at f = −0.25
b.set(xlabel="step k", ylabel="f(θₖ)", ylim=(-0.3, 1), title="Loss per step: a flat region at f ≈ 0")
b.legend(frameon=False, fontsize=9)
plt.tight_layout()
fmt_axes(plt.gcf())
plt.show()

### In many dimensions, saddle points far outnumber minima
At a point where the gradient is 0, each principal direction of curvature (each eigenvalue of the **Hessian
matrix**, the matrix of second derivatives) can be positive or negative. A minimum needs *all* of them to be
positive (section 0.4). The *Deep Learning* book (Chapter 8) compares this to tossing a coin: in $n$ dimensions, you need $n$ heads
in a row. Let's try it with random symmetric matrices, playing the role of an “arbitrary” Hessian:

In [ ]:
rng = np.random.default_rng(0)
DRAWS = 100_000
for n in range(1, 7):
    A = rng.normal(size=(DRAWS, n, n))
    H = (A + A.transpose(0, 2, 1)) / np.sqrt(2)  # a random symmetric matrix (the GOE distribution)
    frac = np.mean((np.linalg.eigvalsh(H) > 0).all(axis=1))
    print(f"n = {n}: all eigenvalues positive in {pct(frac, 3):>8} of the matrices   (coin tosses: {pct(0.5**n, 3)})")
    if n == 2:
        assert abs(frac - (2 - np.sqrt(2)) / 4) < 0.003  # the exact value for n = 2: (2 − √2)/4
print(
    f"({num(DRAWS, 0)} matrices for each n; for n = 2 the exact value is (2 − √2)/4 = {pct((2 - np.sqrt(2)) / 4, 3)})"
)

The fraction falls even faster than the coin tosses, because the eigenvalues of a random matrix repel each other:
for $n = 6$, only 0.001% (1 of the 100,000 matrices tried) have all 6 eigenvalues positive, against 1.562% for the
coin. Dean & Majumdar (2006) computed that this probability decreases like $e^{-0.27\,n^2}$. A neural network has
millions of parameters, so a point with zero gradient in a high-loss region is almost certainly a saddle point
rather than a minimum (Dauphin et al., 2014). This is an argument about random functions; for real networks it is
still a hypothesis under study, but it agrees with what has been measured.

The good news: gradient descent from a random start almost never stops *exactly* at a saddle point (Lee et al.,
2016), because, as we just saw, being off by a tiny bit is enough to escape. The price is **time** spent on the flat
regions around them, and SGD's noise helps shorten it.

## 7 · Softmax regression on MNIST: a hand-derived gradient

Now for a real loss function. The simplest model for the 10 digits is one layer $z = xW + b$ (Lesson 01) with softmax
and cross-entropy (Lesson 04): $W$ of size $784 \times 10$, $b$ of size $10$, 7,850 parameters $\theta = (W, b)$ in
total. With $N$ images stacked as the rows of $X$:

$$\mathcal{L}(W, b) = \frac{1}{N}\sum_{i=1}^{N} -\log \hat y_{i, y_i}, \qquad \hat y_i = \operatorname{softmax}(x_i W + b).$$

Its gradient is surprisingly compact (section 0.5). For one image, $\ell = \log\sum_j e^{z_j} - z_y$, so
$\partial \ell/\partial z_k = \hat y_k - y_k$ ($y$ is the one-hot label): the predicted probability minus the answer.
Since $z_k = \sum_i x_i W_{ik} + b_k$, each $W_{ik}$ receives the share $x_i\,(\hat y_k - y_k)$. Averaged over the
$N$ images, in matrix form:

$$\nabla_W \mathcal{L} = \frac{1}{N} X^\top (\hat Y - Y), \qquad \nabla_b \mathcal{L} = \frac{1}{N}\sum_{i} (\hat y_i - y_i).$$

This is one step of the chain rule; Lesson 06 does the same for multilayer networks. The function `ce` below returns
the loss and $\partial\mathcal{L}/\partial Z = (\hat Y - Y)/N$; `softmax_loss_grad` then multiplies by $X^\top$.

In [ ]:
def ce(Z, y):
    """The mean cross-entropy of the rows of logits Z, and ∂L/∂Z = (Ŷ − Y)/N."""
    Z = Z - Z.max(axis=1, keepdims=True)  # subtract each row's max: same softmax, exp can't overflow (Lesson 04)
    E = np.exp(Z)
    S = E.sum(axis=1, keepdims=True)
    n = len(Z)
    loss = np.mean(np.log(S[:, 0]) - Z[np.arange(n), y])
    D = E / S  # Ŷ: the softmax probabilities, one row per image
    D[np.arange(n), y] -= 1  # Ŷ − Y
    return loss, D / n


def softmax_loss_grad(W, b, X, y):
    loss, D = ce(X @ W + b, y)
    return loss, X.T @ D, D.sum(axis=0)  # ∇W = Xᵀ(Ŷ − Y)/N, ∇b = the column sums


def full_loss(W, b):
    return ce(Xtr @ W + b, ytr)[0]  # the loss on all 60,000 training images


def accuracy(W, b):
    return np.mean((Xte @ W + b).argmax(axis=1) == yte)  # the fraction correct on the 10,000 test images


W0, b0 = np.zeros((784, 10)), np.zeros(10)
print(
    f"{num(W0.size + b0.size, 0)} parameters; at W = 0, b = 0: loss = {num(full_loss(W0, b0), 4)} = ln 10, {pct(accuracy(W0, b0))} correct"
)
assert W0.size + b0.size == 7850 and np.isclose(full_loss(W0, b0), np.log(10))

At $W = 0$, every digit gets a probability of 10%, so the loss is exactly $\ln 10$ (Lesson 04). Before using the
gradient formula, we check it against central differences on **all 7,850 parameters**, at a random $W$ and on 100
images (15,700 loss evaluations):

In [ ]:
rng = np.random.default_rng(1)
Wr, br = rng.normal(0, 0.05, (784, 10)), rng.normal(0, 0.05, 10)
Xs, ys = Xtr[:100], ytr[:100]
_, gW, gb = softmax_loss_grad(Wr, br, Xs, ys)
g_formula = np.r_[gW.ravel(), gb]
theta_r = np.r_[Wr.ravel(), br]  # the 7,850 parameters laid out as one vector: W first, then b
g_numeric = num_grad(lambda th: ce(Xs @ th[:7840].reshape(784, 10) + th[7840:], ys)[0], theta_r)
rel = np.abs(g_numeric - g_formula).max() / np.abs(g_formula).max()
print(
    f"{num(g_formula.size, 0)} partial derivatives: largest difference {sci(np.abs(g_numeric - g_formula).max())}, relative {sci(rel)}"
)
assert rel < 1e-6, "the gradient formula is wrong"
print(f"({num(2 * g_formula.size, 0)} loss evaluations for the finite differences)")

The formula matches the finite differences to about 9 digits. From here on we trust `softmax_loss_grad` and use it;
`num_grad`, on the other hand, is too slow for training (each gradient costs 15,700 loss evaluations instead of one).

## 8 · Gradient descent on the whole training set: which learning rate is too large?

Section 5 says the threshold is $\eta < 2/\lambda_{\max}$, where $\lambda_{\max}$ is the largest curvature. With 7,850
parameters, the curvature along a direction $V$ can be measured with the very same finite-difference trick as in
section 3, applied to the gradient:

$$\text{curvature along } V \approx \Big\langle \frac{\nabla\mathcal{L}(\theta + \varepsilon V) - \nabla\mathcal{L}(\theta - \varepsilon V)}{2\varepsilon},\, V \Big\rangle, \qquad \|V\| = 1.$$

At $W = 0$ there is a shortcut (section 0.5): every image has the same $\hat y_k = 0.1$, so the curvature splits into a product of
two parts: the part over images is an eigenvalue of the matrix $M = \tilde X^\top \tilde X / N$ (with
$\tilde x = [x, 1]$, since $b$ is the weight of an input that is always 1), and the part over the 10 classes is
$0.1$. So $\lambda_{\max} = 0.1 \times$ the largest eigenvalue of $M$. The cell below computes it, then measures the
curvature along exactly that direction with finite differences, as a check.

In [ ]:
Xb = np.c_[Xtr, np.ones(N)]  # x̃ = [x, 1]
eigval, eigvec = np.linalg.eigh(Xb.T @ Xb / N)
lam_max = 0.1 * eigval[-1]
u = eigvec[:, -1]  # the most curved "image" direction
c = np.array([1.0, -1.0] + [0.0] * 8) / np.sqrt(2)  # a "class" direction: raise the z of digit 0, lower that of digit 1
VW, Vb = np.outer(u[:784], c), u[784] * c  # a direction of length 1 in the space of the 7,850 parameters
eps = 1e-4
_, gWp, gbp = softmax_loss_grad(W0 + eps * VW, b0 + eps * Vb, Xtr, ytr)
_, gWm, gbm = softmax_loss_grad(W0 - eps * VW, b0 - eps * Vb, Xtr, ytr)
curv = (np.sum((gWp - gWm) * VW) + np.sum((gbp - gbm) * Vb)) / (2 * eps)
print(
    f"largest eigenvalue of M = {num(eigval[-1], 4)}  →  λ_max = {num(lam_max, 4)},  threshold 2/λ_max = {num(2 / lam_max, 3)}"
)
print(f"curvature along that direction, measured with finite differences: {num(curv, 4)}")
assert abs(curv - lam_max) / lam_max < 1e-6

At $W = 0$, the threshold is $\eta \approx 0.513$. We run gradient descent on **all 60,000 images** at every step,
for 20 steps, with three learning rates: just below the threshold ($0.5$), twice that, and four times that.

In [ ]:
def gd_full(eta, steps=20):
    """Gradient descent on the whole training set, from W = 0. Returns W, b and the loss before each step (and after
    the last one)."""
    W, b = np.zeros((784, 10)), np.zeros(10)
    losses = []
    for _ in range(steps):
        loss, gW, gb = softmax_loss_grad(W, b, Xtr, ytr)  # the gradient on all 60,000 images
        losses.append(loss)
        W -= eta * gW
        b -= eta * gb
    losses.append(full_loss(W, b))
    return W, b, np.array(losses)


gd_runs = {eta: gd_full(eta) for eta in [0.5, 1.0, 2.0]}
for eta, (W_, b_, Ls) in gd_runs.items():
    up = int(np.sum(np.diff(Ls) > 0))
    print(
        f"η = {num(eta, 1)}: loss {num(Ls[0], 3)} → {num(Ls[-1], 3)} after 20 steps, highest {num(Ls.max(), 3)} (step {Ls.argmax()}), "
        f"{up} steps uphill; test accuracy {pct(accuracy(W_, b_))}"
    )
L05, L2 = gd_runs[0.5][2], gd_runs[2.0][2]
assert np.all(np.diff(L05) < 0) and L2.max() > np.log(10)

The figure below shows the loss on the whole training set after each step, for the three learning rates (circles,
squares, triangles); the dashed line is $\ln 10$, the loss at the start.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
for (eta, (_, _, Ls)), color, m, ls in zip(gd_runs.items(), [BLUE, GREEN, ORANGE], ["o", "s", "^"], ["-", "--", ":"]):
    ax.plot(Ls, marker=m, ls=ls, color=color, lw=1.8, ms=5, label=f"η = {num(eta, 1)}")
ax.axhline(np.log(10), color=GREY, ls="--", lw=1)
ax.set(xlabel="step (each step uses all 60,000 images)", ylabel="training loss", xticks=range(0, 21, 2))
ax.set_title(f"Full-batch gradient descent; threshold at W = 0: 2/λ_max = {num(2 / lam_max, 3)}", fontsize=11)
ax.legend(frameon=False)
fmt_axes(plt.gcf())
plt.show()

With $\eta = 0.5$, just below the threshold, the loss decreases at every step. At twice or four times the threshold,
there are steps where the loss *increases*: with $\eta = 2$ it shoots up to almost three times its starting value.
But unlike the bowl of section 5, it doesn't diverge forever; it comes back down. One reason: cross-entropy is not a
bowl. Far from the bottom it grows only roughly linearly, so its gradient is bounded: for any logits,
$\|\hat y - y\| \le \sqrt{2}$, so the gradient of one image is never longer than $\sqrt{2}\,\|\tilde x\|$. Each step
moves $\theta$ by a bounded amount, instead of being multiplied up forever as in the bowl (or as with the demo's
quartic function, whose gradient grows like $\theta^3$). The curvature also changes from place to place, so the
threshold $2/\lambda$ only holds **near one point**. For real neural networks, Cohen et al. (2021) found that
full-batch gradient descent often brings itself to exactly that edge: the largest curvature rises gradually, then
hovers around $2/\eta$ (they call it the *edge of stability*).

There is a bigger problem: the 20 steps above computed the gradient on $20 \times 60{,}000$ images, and the loss is
still 0.602 with $\eta = 0.5$. The next section does it differently: each step looks at only a handful of images.

## 9 · Mini-batch SGD: steps and epochs

**Mini-batch SGD** (stochastic gradient descent on small batches) replaces the gradient on the whole training set
with the gradient on $B$ randomly chosen images (section 0.6). The standard recipe (also the one the course's
`reel/mnist_mlp/train.py` uses to train the MLP): each **epoch** shuffles the order of the 60,000 images, then cuts it
into consecutive batches, one batch per **step (iteration)**. Every image is used exactly once per epoch.

- Gradient descent on the whole training set: 1 step = 1 epoch.
- SGD with $B = 64$: 1 epoch = $\lceil 60{,}000 / 64 \rceil$ steps, and the last batch holds only the remainder.

In [ ]:
def sgd(eta, B=64, epochs=1, seed=0, checkpoints=()):
    """Mini-batch SGD for softmax regression, from W = 0. Returns W, b and [(step, loss on the whole training set)] at
    the checkpoints."""
    W, b = np.zeros((784, 10)), np.zeros(10)
    rng = np.random.default_rng(seed)
    curve, step = [], 0
    for _ in range(epochs):
        order = rng.permutation(N)  # every epoch reshuffles the order of the images
        for i in range(0, N, B):
            if step in checkpoints:
                curve.append((step, full_loss(W, b)))
            idx = order[i : i + B]  # one batch: the next B images in the shuffled order
            _, gW, gb = softmax_loss_grad(W, b, Xtr[idx], ytr[idx])
            W -= eta * gW
            b -= eta * gb
            step += 1
    curve.append((step, full_loss(W, b)))
    return W, b, curve


steps_per_epoch = len(range(0, N, 64))
print(f"B = 64: {steps_per_epoch} steps per epoch; the last batch has {N - 64 * (steps_per_epoch - 1)} images")
assert steps_per_epoch == 938

### 🤔 Predict first
Same $\eta = 0.5$, same **20 steps**. Gradient descent on the whole training set uses 60,000 images per step; SGD
uses 64 images per step. After 20 steps, which one has the lower training loss? And once each has used *the same
number of images*, one epoch?

In [ ]:
cps = set(range(21)) | {int(k) for k in np.geomspace(1, 937, 14)}
W_sgd, b_sgd, curve = sgd(0.5, checkpoints=cps)
sgd_loss = dict(curve)
print(f"after 20 steps:  full-batch GD {num(L05[20], 3)}  ·  SGD (B = 64) {num(sgd_loss[20], 3)}")
print(f"after 1 epoch:   full-batch GD {num(L05[1], 3)} (1 step)  ·  SGD (B = 64) {num(sgd_loss[938], 3)} (938 steps)")
print(
    f"GD needs 20 epochs to get down to {num(L05[20], 3)}; SGD is at {num(sgd_loss[938], 3)} after 1 epoch, test accuracy {pct(accuracy(W_sgd, b_sgd))}"
)
assert L05[20] < sgd_loss[20] and sgd_loss[938] < L05[20]
assert abs(accuracy(W_sgd, b_sgd) - 0.9172) < 0.0005

Counted in **steps**, gradient descent on the whole training set wins slightly: 0.602 against 0.694, because each of
its steps goes in exactly the steepest direction. But each of those steps costs 938 times as much. Counted in
**images used**, which is what determines the run time, SGD wins by far: after one epoch it is already down to
0.296, lower than even 20 epochs of gradient descent.

The figure below shows both views. Left: the loss per step. Right: the same data against the number of images used
to compute gradients (log axis). Gradient descent is the blue line with circles, SGD the orange line with squares.

In [ ]:
sgd_steps = np.array(sorted(sgd_loss))
sgd_vals = np.array([sgd_loss[k] for k in sgd_steps])
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
a.plot(range(21), L05, "-o", color=BLUE, ms=4, label="full-batch GD")
a.plot(sgd_steps[sgd_steps <= 20], sgd_vals[sgd_steps <= 20], "--s", color=ORANGE, ms=4, label="SGD, B = 64")
a.set(xlabel="step", ylabel="training loss", title="Per step", xticks=range(0, 21, 5))
b.semilogx(np.arange(1, 21) * N, L05[1:], "-o", color=BLUE, ms=4, label="full-batch GD")
b.semilogx(sgd_steps[1:] * 64, sgd_vals[1:], "--s", color=ORANGE, ms=4, label="SGD, B = 64")
b.axvline(N, color=GREY, lw=1, ls=":")
b.text(N * 1.1, 2.1, "1 epoch", fontsize=9)
b.set(xlabel="images used to compute gradients", title="Per amount of computation")
a.legend(frameon=False)
plt.tight_layout()
fmt_axes(plt.gcf())
plt.show()

Why SGD is so efficient: the gradient on the whole training set is an average of 60,000 per-image gradients, and
MNIST images are highly redundant (thousands of 1s are nearly identical). A batch of 64 images already gives a good
enough estimate of that average, at a cost 938 times lower. Section 10 measures just how good “good enough” is.

### Learning rates for SGD
A sweep of $\eta$ with SGD, $B = 64$, one epoch, seed 0:

In [ ]:
for eta in [0.001, 0.01, 0.1, 0.5, 1.0, 100.0]:
    W_, b_, cv = sgd(eta)
    print(f"η = {num(eta, 3):>7}: training loss {num(cv[-1][1], 3):>6}, test accuracy {pct(accuracy(W_, b_))}")
    if eta == 0.1:
        assert abs(accuracy(W_, b_) - 0.9089) < 0.0005
    if eta == 100.0:
        assert cv[-1][1] > np.log(10) and accuracy(W_, b_) > 0.7  # a loss worse than at the start, a decent accuracy
seed_acc = [accuracy(W_sgd, b_sgd)] + [accuracy(*sgd(0.5, seed=s)[:2]) for s in (1, 2)]  # seed 0: section 9's model
print(f"η = 0.5 with 3 seeds: test accuracy {pct(min(seed_acc))} – {pct(max(seed_acc))}")

- **Too small** ($0.001$): one epoch only brings the loss from 2.303 down to 1.587. The right direction, but at a
  crawl.
- **The good range** ($0.5$ to $1$): loss 0.296–0.298, 91.44%–91.72% correct. The difference between $\eta = 0.5$
  and $1$ lies within the spread between seeds (91.34% – 91.72% with $\eta = 0.5$).
- **Too large** ($\eta = 100$): the loss at the end of the epoch is even higher than at the start ($\ln 10$), and yet
  the accuracy is still decent. Each step jumps so far that the logits become very large: softmax puts almost all of
  the probability on one digit, and the model is **confidently wrong** on many images, exactly the kind of error
  cross-entropy penalizes most heavily (Lesson 04). Then $\hat Y - Y$ comes down to “subtract the image from the
  digit it was mistaken for, add it to the correct digit”: the perceptron learning rule of Lesson 01. With an $\eta$
  that large, the result also varies with the seed and even with the machine's rounding errors, so the numbers
  printed on your machine may differ.

Looking at the accuracy isn't enough to choose $\eta$; you have to look at the loss too.

## 10 · How good is the gradient of a batch?

The batch gradient $g_B$ is the average of the gradients of $B$ randomly chosen images, so its mean is exactly the
true gradient $g$ (an unbiased estimate, section 0.6); there is only noise. By the standard-error law for a sample mean, the
typical deviation $\|g_B - g\|$ shrinks like $1/\sqrt{B}$: a batch 100 times larger cuts the noise only 10-fold.

For softmax regression, the gradient of each image is the outer product $\tilde x_i^\top(\hat y_i - y_i)$, so its
squared length is the product $\|\tilde x_i\|^2\,\|\hat y_i - y_i\|^2$. This lets us compute the *exact* variance of
the noise over all 60,000 images, with no sampling, and compare it with measurements on 100 random batches of each
size.

In [ ]:
SIZES_B = [1, 4, 16, 64, 256, 1024, 4096]


def batch_noise(W, b, draws=100, seed=2):
    """The relative deviation ‖g_B − g‖/‖g‖ (measured and theoretical) and cos(g_B, g), for each batch size B."""
    _, D = ce(Xtr @ W + b, ytr)
    g = np.r_[(Xtr.T @ D).ravel(), D.sum(axis=0)]  # the true gradient, on all 60,000 images
    sq = (np.sum(Xtr**2, axis=1) + 1) * np.sum((N * D) ** 2, axis=1)  # ‖gᵢ‖² = ‖x̃ᵢ‖² · ‖ŷᵢ − yᵢ‖²
    sigma2 = sq.mean() - g @ g  # the variance of one image's gradient
    rng = np.random.default_rng(seed)
    out = []
    for B in SIZES_B:
        err, cos = [], []
        for _ in range(draws):
            idx = rng.choice(N, B, replace=False)
            _, bW, bb = softmax_loss_grad(W, b, Xtr[idx], ytr[idx])
            gB = np.r_[bW.ravel(), bb]
            err.append(np.sum((gB - g) ** 2))
            cos.append(gB @ g / (np.linalg.norm(gB) * np.linalg.norm(g)))
        theory = np.sqrt(sigma2 / B * (N - B) / (N - 1)) / np.linalg.norm(g)  # sampling without replacement
        out.append((B, np.sqrt(np.mean(err)) / np.linalg.norm(g), theory, np.mean(cos)))
    return np.array(out)


noise0 = batch_noise(W0, b0)
for B, rel, th, cos in noise0:
    print(
        f"W = 0,  B = {int(B):>4}: ‖g_B − g‖/‖g‖ measured {num(rel, 3):>6}, theory {num(th, 3):>6};  cos(g_B, g) = {num(cos, 2)}"
    )
assert np.all(np.abs(noise0[2:, 1] / noise0[2:, 2] - 1) < 0.1)  # B ≥ 16: the measurement matches the theory within 10%

At the start, a batch of 64 images already gives a gradient that is off by about the length of the true gradient
itself, yet still points the same way (cos 0.69). With 100 batches of each size, the measurement matches the theory
to within a few percent at every $B$.

### Checking section 0.6: the average of many batch gradients
The measurement above matches the variance of section 0.6. As for unbiasedness, $\mathbb{E}[g_B] = g$: if it holds,
the average $\bar g_k$ of $k$ independently drawn batch gradients must approach $g$, with a deviation that shrinks like
$1/\sqrt{k}$ times the deviation of one batch (section 0.6); if $g_B$ were biased, that deviation would level off at
the size of the bias. The cell below draws 1,000 batches of 64 images at $W = 0$ and prints the relative deviation
$\lVert\bar g_k - g\rVert/\lVert g\rVert$ after $k$ = 1, 10, 100 and 1,000 batches, next to the theoretical value: the
relative deviation of one batch from the cell above, divided by $\sqrt{k}$.

In [ ]:
_, D0 = ce(Xtr @ W0 + b0, ytr)
g_true = np.r_[(Xtr.T @ D0).ravel(), D0.sum(axis=0)]  # the true gradient at W = 0, as in batch_noise
th64 = noise0[3, 2]  # the theoretical relative deviation of one batch of 64 images (the cell above)
rng = np.random.default_rng(2)
total, k_done = np.zeros_like(g_true), 0
for k in [1, 10, 100, 1000]:
    while k_done < k:
        idx = rng.choice(N, 64, replace=False)
        _, bW, bb = softmax_loss_grad(W0, b0, Xtr[idx], ytr[idx])
        total += np.r_[bW.ravel(), bb]
        k_done += 1
    dev = np.linalg.norm(total / k - g_true) / np.linalg.norm(g_true)
    print(
        f"k = {num(k, 0):>5}:  the average is off by {num(dev, 3)} times ‖g‖,  theory {num(th64 / np.sqrt(k), 3)} times ‖g‖"
    )
    assert 0.7 < dev / (th64 / np.sqrt(k)) < 1.4  # the 1/√k rate: no sign of a bias
assert abs(th64 - 1.065732) < 1e-4 and abs(np.linalg.norm(g_true) - 1.040932) < 1e-4, (
    "differs from the lesson page: check the data"
)

Each time the number of batches grows tenfold, the deviation shrinks by about $\sqrt{10}$, close to the theoretical
value, and it doesn't level off anywhere: the average of 1,000 batches is off by only 0.029 times the length of $g$,
against 0.904 for a single batch. That is what an unbiased estimate looks like: the noise of the individual batches
cancels out as they are averaged. A bias, if there is one, is smaller than that last level.

### 🤔 Predict first
Now take the model after one epoch of SGD from section 9 (loss 0.296). There, does the gradient of a 64-image batch
still point roughly the same way as the true gradient? Guess their cosine: close to 1, about 0.7 as at the start, or
much smaller?

In [ ]:
noise1 = batch_noise(W_sgd, b_sgd)
for B, rel, th, cos in noise1:
    print(
        f"after 1 epoch, B = {int(B):>4}: ‖g_B − g‖/‖g‖ measured {num(rel, 2):>6}, theory {num(th, 2):>6};  cos(g_B, g) = {num(cos, 2)}"
    )
assert np.all(np.abs(noise1[2:, 1] / noise1[2:, 2] - 1) < 0.1)
assert noise1[3, 3] < 0.3  # B = 64: the batch gradient is almost orthogonal to the true gradient

Much smaller: the cosine is only about 0.15, and the deviation is almost 6 times the length of the true gradient.
(For $B = 1$ and $4$, the measurements are below the theory: after training, a few hard images have very large
gradients, and 100 batches rarely happen to draw them.) Near the bottom, the true gradient gets small (the images
“pull” in different directions and nearly cancel each other out), but the per-image gradients don't, so **the noise
drowns out the signal**.

SGD still makes it downhill because the noise averages to 0: over $k$ steps, the signal adds up in proportion to $k$,
while the noise grows only like $\sqrt{k}$. But with a fixed $\eta$, it never settles exactly at the bottom; it
hovers around it. Robbins & Monro (1951) showed that, to converge, $\eta$ must decrease so that
$\sum_k \eta_k = \infty$ and $\sum_k \eta_k^2 < \infty$ (for example $\eta_k \propto 1/k$). Learning-rate decay
schedules and optimizers such as momentum and Adam are the subject of Lesson 07.

The figure below shows the relative deviation against the batch size (both axes logarithmic): dots are measurements,
lines are theory; blue circles at $W = 0$, orange squares after one epoch. The two lines are parallel, with slope
$-\tfrac{1}{2}$, exactly the $1/\sqrt{B}$ law.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
for res, color, m, name in [(noise0, BLUE, "o", "at W = 0"), (noise1, ORANGE, "s", "after 1 epoch of SGD")]:
    ax.loglog(res[:, 0], res[:, 2], color=color, lw=1.5)
    ax.loglog(res[:, 0], res[:, 1], m, color=color, ms=7, label=name)
ax.axhline(1, color=GREY, ls="--", lw=1)
ax.text(1.1, 0.8, "noise = signal", ha="left", fontsize=9)
ax.set(xlabel="batch size B", ylabel="‖g_B − g‖ / ‖g‖", title="The noise of the batch gradient shrinks like 1/√B")
ax.legend(frameon=False)
fmt_axes(plt.gcf())
plt.show()

## 11 · The same loop for the course's network: 784 → 128 → 64 → 10

The course uses one running example: MLSysBook's 784 → 128 → 64 → 10 MLP (ReLU, softmax + cross-entropy, He
initialization), trained with exactly the mini-batch SGD we just wrote: $\eta = 0.1$, $B = 64$, seed 0, 5 epochs
(`reel/mnist_mlp/train.py`). Nothing in the loop changes; only the function that computes the gradient is more
complex.

The two “backward pass” lines in the cell below are **backpropagation**, the subject of Lesson 06. We don't derive
them here; we only do what section 3 taught: never trust a gradient formula before checking it with finite
differences.

In [ ]:
SIZES = [784, 128, 64, 10]


def mlp_init(seed=0):
    """He initialization as in train.py; also returns the rng, because train.py goes on using it to shuffle the
    images."""
    rng = np.random.default_rng(seed)
    params = [
        [rng.normal(0, np.sqrt(2 / n_in), (n_in, n_out)), np.zeros(n_out)] for n_in, n_out in itertools.pairwise(SIZES)
    ]
    return params, rng


def mlp_forward(params, x):
    (W1, b1), (W2, b2), (W3, b3) = params
    h1 = np.maximum(0, x @ W1 + b1)  # z = xW + b, then ReLU (Lessons 02–03)
    h2 = np.maximum(0, h1 @ W2 + b2)
    return h1, h2, h2 @ W3 + b3


def mlp_loss_grad(params, x, y):
    h1, h2, z = mlp_forward(params, x)
    loss, dz = ce(z, y)  # dz = (Ŷ − Y)/N, exactly as in section 7
    (_, _), (W2, _), (W3, _) = params
    # backward pass: Lesson 06 derives these two lines with the chain rule; here we only check them by finite differences
    dh2 = (dz @ W3.T) * (h2 > 0)
    dh1 = (dh2 @ W2.T) * (h1 > 0)
    return loss, [(x.T @ dh1, dh1.sum(0)), (h1.T @ dh2, dh2.sum(0)), (h2.T @ dz, dz.sum(0))]


params, _ = mlp_init(seed=0)
n_params = sum(W.size + b.size for W, b in params)
print(
    f"{num(n_params, 0)} parameters; a gradient by finite differences takes {num(2 * n_params, 0)} loss evaluations per step"
)
assert n_params == 109386

Check 30 randomly chosen partial derivatives (5 entries of each $W$ and each $b$ in the three layers), on 100 images:

In [ ]:
def loss_with(P, idx, value, x, y):
    """The loss with exactly one parameter set, P[idx] = value (then the old value is put back)."""
    old = P[idx]
    P[idx] = value
    loss = mlp_loss_grad(params, x, y)[0]
    P[idx] = old
    return loss


xs, ys = Xtr[:100], ytr[:100]
_, grads = mlp_loss_grad(params, xs, ys)
rng = np.random.default_rng(3)
worst = 0.0
for layer, which in itertools.product(range(3), range(2)):  # which = 0: W, 1: b
    P = params[layer][which]
    for _ in range(5):
        idx = tuple(int(rng.integers(0, n)) for n in P.shape)
        g_num = num_grad(lambda v, P=P, idx=idx: loss_with(P, idx, v[0], xs, ys), [P[idx]])[0]
        worst = max(worst, abs(g_num - grads[layer][which][idx]))
print(f"30 partial derivatives: largest difference between the formula and the finite differences {sci(worst)}")
assert worst < 1e-7

The backward pass matches. Now run exactly one epoch of `train.py`: the same initialization, the same order of
images, recording the mean loss of every 50 batches as the course's data file does (`web/src/content/mnist-mlp.json`,
key `history`), then compare with the values in it.

In [ ]:
JSON_HISTORY = [1.2031, 0.5253, 0.4288, 0.3964, 0.3448, 0.328, 0.3182, 0.2595, 0.2644]  # mnist-mlp.json, steps 50–450
JSON_HISTORY += [0.262, 0.2222, 0.2438, 0.2219, 0.2178, 0.1976, 0.2091, 0.2098, 0.2072]  # steps 500–900
params, rng = mlp_init(seed=0)
order = rng.permutation(N)  # epoch 1
batch_losses = []
t0 = time.perf_counter()
for i in range(0, N, 64):
    idx = order[i : i + 64]
    loss, grads = mlp_loss_grad(params, Xtr[idx], ytr[idx])
    batch_losses.append(loss)
    for layer, (gW, gb) in zip(params, grads):
        layer[0] -= 0.1 * gW  # θ ← θ − η∇L, for each matrix W
        layer[1] -= 0.1 * gb  # and each vector b
hist = np.array(batch_losses[: 50 * (len(batch_losses) // 50)]).reshape(-1, 50).mean(axis=1)
acc_mlp = np.mean(mlp_forward(params, Xte)[2].argmax(axis=1) == yte)
print(
    f"{len(batch_losses)} steps in {num(time.perf_counter() - t0, 1)} seconds; test accuracy {pct(acc_mlp)} after 1 epoch"
)
print("loss every 50 steps:", " · ".join(num(v, 4) for v in hist[:6]), "…", num(hist[-1], 4))
print(f"largest difference from mnist-mlp.json: {num(np.abs(np.round(hist, 4) - JSON_HISTORY).max(), 4)}")
assert np.abs(hist - JSON_HISTORY).max() < 5e-4 and abs(acc_mlp - 0.9427) < 0.0005, "differs from the course's data"

Every number matches the course: after one epoch, the network gets 94.27% of the test images right (`train.py` runs
4 more epochs to reach 97.27%, the `testAcc` key of `mnist-mlp.json`, the number the demos use from Lesson 04 on). In
the same single epoch, the softmax regression of section 9 reaches only 91.72%: it has just one linear layer.

The figure below shows the loss of **each batch** (thin gray line, 938 steps) and the average every 50 steps (blue
dots). Each batch has only 64 images, so its loss jumps up and down sharply around the downward trend: that is the
SGD noise that section 10 measured.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(np.arange(1, len(batch_losses) + 1), batch_losses, color=GREY, lw=0.6, label="loss of each batch (64 images)")
ax.plot(np.arange(50, 50 * len(hist) + 1, 50), hist, "o", color=BLUE, ms=6, label="average every 50 steps")
ax.set(xlabel="step (1 epoch = 938 steps)", ylabel="cross-entropy", ylim=(0, 2.6), xlim=(0, 940))
ax.set_title("The 784 → 128 → 64 → 10 network, one epoch of SGD (η = 0.1, B = 64, seed 0)", fontsize=11)
ax.legend(frameon=False)
fmt_axes(plt.gcf())
plt.show()

## 12 · Try changing things

The cells below rerun the same functions with other settings. You can change the values and run them again.

### The starting point decides the valley
With the demo's function and $\eta = 0.15$, change $\theta_0$:

In [ ]:
for start in [-3.0, -1.0, 0.1, 0.18, 0.2, 0.5, 3.0]:
    tr, st = gd_1d(0.15, start, max_steps=500)
    where = "global" if tr[-1] < 0 else "local"
    print(
        f"θ₀ = {num(start, 2):>5}: {st} after {len(tr) - 1:>3} steps at θ = {num(tr[-1], 3):>6} (the {where} minimum)"
    )

The boundary between the two valleys is the peak at $\theta \approx 0.189$: starting to the left of the peak leads to
the global minimum, starting to the right leads to the local minimum. The closer the start is to the peak, the closer
the slope is to 0, and so the more steps it takes.

### Batch size, with the same $\eta = 0.5$ and one epoch

In [ ]:
for B in [8, 16, 64, 256, 1024]:
    W_, b_, cv = sgd(0.5, B=B)
    print(
        f"B = {B:>4}: {num(cv[-1][0], 0):>5} steps per epoch, loss {num(cv[-1][1], 3)}, test accuracy {pct(accuracy(W_, b_))}"
    )

A large batch gives a less noisy gradient, but one epoch then has fewer steps, so for the same amount of computation
it gets less far. With a batch that is too small at $\eta = 0.5$, the noise is too large: each step moves the
parameters by about $\eta$ times the deviation of the batch gradient, and that deviation scales like $1/\sqrt{B}$.
In this range, $B = 64$ is the best (with seed 0).

### Larger batches, longer steps
Goyal et al. (2017) trained on ImageNet with batches of 8,192 images using the **linear scaling rule**: multiply $B$
by $k$, and multiply $\eta$ by $k$ too, so that one epoch still covers a similar distance. Let's try it from $B = 64$,
$\eta = 0.1$:

In [ ]:
for k in [1, 4, 16, 64]:
    W_, b_, cv = sgd(0.1 * k, B=64 * k)
    print(
        f"B = {64 * k:>4}, η = {num(0.1 * k, 1):>3}: {cv[-1][0]:>3} steps, loss {num(cv[-1][1], 3)}, test accuracy {pct(accuracy(W_, b_))}"
    )

The rule keeps the result up to $B = 256$ and nearly up to $1{,}024$, then breaks at $B = 4{,}096$: there
$\eta = 6.4$ is more than ten times the threshold $2/\lambda_{\max} \approx 0.51$ of section 8, and one epoch has only
15 steps left in which to recover. Goyal et al. also had to add *warmup* (increasing $\eta$ gradually over the first
few epochs) to make large batches work.

## 13 · The limit: where does the gradient of a multilayer network come from?

Gradient descent needs only one thing: $\nabla\mathcal{L}(\theta)$. For softmax regression, we could write it by hand
in one line ($X^\top(\hat Y - Y)/N$) because there is only one layer. For the 784 → 128 → 64 → 10 network, we used
two “backward pass” lines without explaining them, and could only check them with finite differences: computing the
whole gradient by finite differences would take 218,772 loss evaluations for *every step*, against one forward pass
and one backward pass. Lesson 06 derives those lines from the chain rule (section 0.7).

There is one more limit: with a fixed $\eta$, SGD never settles exactly at the bottom because of the noise
(section 10), and a single $\eta$ shared by every direction is slow in a narrow valley (section 5). Momentum, Adam
and learning-rate decay schedules are the subject of Lesson 07.

## Summary
- The gradient is the vector of partial derivatives, which central differences can measure; use them to check every
  formula (sections 3, 7, 11).
- One step: $\theta \leftarrow \theta - \eta\,\nabla\mathcal{L}(\theta)$; for a small $\eta$ it decreases
  $\mathcal{L}$ by exactly $\eta\lVert\nabla\mathcal{L}\rVert^2$ to first order. On the demo's function, from
  $\theta_0 = 2.6$: $\eta = 0.02$ needs 126 steps, $0.15$ needs 13, $0.9$ jumps back and forth forever, $1.3$ flies
  out after 3 steps.
- Near a bottom with curvature $\lambda$, each step multiplies the distance to the bottom by $1 - \eta\lambda$
  (measured to 4 digits, sign included), so gradient descent is stable when $\eta < 2/\lambda$ (0.551 and 0.734 at the
  demo's two bottoms; 0.513 for softmax regression at $W = 0$). A narrow valley ($\kappa = 10$) forces a zig-zag:
  69 steps at the best $\eta$.
- The gradient points in the locally steepest direction, not at the bottom (54.2° off in the narrow bowl), and it is
  0 at saddle points too. In many dimensions, saddle points far outnumber minima.
- Mini-batch SGD: one epoch with $B = 64$ is 938 cheap steps and brings the loss down to 0.296; it takes 20 epochs of
  gradient descent on the whole training set to get down to 0.602.
- The gradient of a batch is unbiased (the average of 1,000 batches is off by only 0.029 times $\lVert g\rVert$), its
  noise shrinks like $1/\sqrt{B}$, and near the bottom it is larger than the gradient itself (cos ≈ 0.15 with
  $B = 64$).
- The same loop trains the course's 784 → 128 → 64 → 10 network: 94.27% after one epoch, exactly as in the course's
  data.

## Review questions
1. Why do we step along $-\nabla\mathcal{L}$ and not $+\nabla\mathcal{L}$?
2. Near a bottom with curvature $\lambda = 4$, what is the largest learning rate that is still stable? With
   $\eta = 0.4$, how does $\theta$ approach the bottom?
3. Why, in the bowl $\lambda = (1, 10)$, is the best $\eta$ not $1/\lambda_2 = 0.1$?
4. If gradient descent starts exactly at $(2, 0)$ on the saddle function, where does it stop? Why does a little
   noise help?
5. How many steps is one epoch of SGD with $B = 64$ on 60,000 images? And of gradient descent on the whole training
   set?
6. If the batch is 100 times larger, how many times smaller is the noise of the batch gradient?

<details><summary>Hints and answers</summary>

1. $\nabla\mathcal{L}$ points in the direction in which the loss increases fastest; the opposite direction is the one
   in which it decreases fastest (for a small enough step; sections 0.1 and 0.2).
2. $2/\lambda = 0.5$. With $\eta = 0.4$, the factor is $1 - \eta\lambda = -0.6$: the distance to the bottom shrinks to
   60% at each step but changes sign, so $\theta$ jumps back and forth on either side of the bottom as it closes in
   (section 0.3; the check in section 5 measures exactly this regime with $\eta = 0.5$).
3. With $\eta = 0.1$, the gentle direction shrinks only by a factor of $0.9$ per step. $\eta = 2/11$ balances the two
   directions (the same factor in magnitude, $0.818$), so the slowest direction goes faster.
4. At the saddle point $(0, 0)$: the $\theta_2$ component of the gradient is always 0 on the line $\theta_2 = 0$, so
   nothing pushes it off. Noise makes $\theta_2$ nonzero, and the negative curvature in that direction amplifies it
   until it escapes.
5. $\lceil 60{,}000/64 \rceil = 938$ steps. Gradient descent on the whole training set: 1 step per epoch.
6. About 10 times: the noise is proportional to $1/\sqrt{B}$.
</details>

## Further reading
- C. Lemaréchal (2012), [Cauchy and the Gradient Method](https://doi.org/10.4171/dms/6/27): Cauchy proposed the method in 1847.
- H. Robbins & S. Monro (1951), [A Stochastic Approximation Method](https://doi.org/10.1214/aoms/1177729586): the origin of SGD.
- Goodfellow, Bengio & Courville, [*Deep Learning*, Chapter 8](https://www.deeplearningbook.org/contents/optimization.html):
  mini-batches, saddle points, flat regions.
- Y. Dauphin et al. (2014), [Identifying and attacking the saddle point problem](https://arxiv.org/abs/1406.2572).
- D. Dean & S. Majumdar (2006), [Large deviations of extreme eigenvalues of random matrices](https://doi.org/10.1103/PhysRevLett.97.160201):
  the probability that every eigenvalue is positive.
- J. Lee et al. (2016), [Gradient Descent Converges to Minimizers](https://arxiv.org/abs/1602.04915): a random initialization
  almost surely avoids saddle points; S. Du et al. (2017), [Gradient Descent Can Take Exponential Time to Escape Saddle Points](https://arxiv.org/abs/1705.10412).
- J. Cohen et al. (2021), [Gradient Descent on Neural Networks Typically Occurs at the Edge of Stability](https://arxiv.org/abs/2103.00065).
- P. Goyal et al. (2017), [Accurate, Large Minibatch SGD](https://arxiv.org/abs/1706.02677): the linear scaling rule and warmup.
- [Dive into Deep Learning, the Gradient Descent section](https://d2l.ai/chapter_optimization/gd.html) and
  [CS231n, Optimization](https://cs231n.github.io/optimization-1/).

**Next lesson:** [Lesson 06 · Backpropagation](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/lan-truyen-nguoc/): derive the
gradient of a multilayer network with the chain rule, at roughly the cost of one forward pass.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")